# Facilitator-Member Difference Identification

## Creating the csv file for the following:

`Name-conference-year`

**Also for the following:**

- facilitator vs. non-facilitator

- ppl-team vs. ppl-not-team

- ppl-funded-team vs. ppl-not-funded-team

## Regression Analysis

Figuring out the question: 

**By analyzing at the individual level, do facilitators act differently than team members?**

Also, considering again, the classification from above:

- facilitator vs. non-facilitator

- ppl-team vs. ppl-not-team

- ppl-funded-team vs. ppl-not-funded-team

# Running the features alone (without existing knowledge)

Determining if people are (or not) facilitator, funded-team, on-team.

**Probably would just run cross-validation on this.**

## Fine-tuning stuff (09/29/2025)

### floating points

In [4]:
import json, pandas as pd, numpy as np
from pathlib import Path
from collections import defaultdict, Counter

# ========= CONFIG =========
DATA_DIR   = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/data")  # root with 2021MZT, 2022SLU, ...
OUTPUT_DIR = Path("/Users/maxchalekson/Desktop/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WRITE_PERSON_SESSION = True  # set False to skip writing the audit csv

# ========= HELPERS =========
def _load_json(fp: Path):
    with open(fp, "r") as f:
        return json.load(f)

def extract_person_metrics_from_session_data(conf_path: Path, session_id: str):
    """
    Compute per-person p_* metrics for a single session from /session_data/<session_id>.json.
    Returns dict: person -> {p_*: value}
    """
    f = conf_path / "session_data" / f"{session_id}.json"
    if not f.exists():
        return {}

    data = _load_json(f)
    rows = data.get("all_data", [])
    by_person = defaultdict(lambda: Counter())
    for r in rows:
        speaker = r.get("speaker")
        if not speaker:
            continue
        dur = r.get("speaking_duration", 0) or 0
        by_person[speaker]["p_speaking_duration_sec"] += float(dur)
        by_person[speaker]["p_turns"] += 1

        if str(r.get("interuption","")).strip().lower() == "yes":
            by_person[speaker]["p_interruptions_made"] += 1
        if str(r.get("overlap","")).strip().lower() == "yes":
            by_person[speaker]["p_overlaps"] += 1
        if str(r.get("screenshare","")).strip().lower() == "yes":
            by_person[speaker]["p_screenshare_segments"] += 1

        by_person[speaker]["p_smile_self_total"]  += float(r.get("smile_self", 0) or 0)
        by_person[speaker]["p_smile_other_total"] += float(r.get("smile_other", 0) or 0)
        by_person[speaker]["p_nods_received"]     += float(r.get("nods_others", 0) or 0)

        # OPTIONAL: collect annotation counts/scores at utterance level if present
        ann = r.get("annotations") or {}
        for cat, info in ann.items():
            # normalize category name to snake-ish
            key = (
                cat.lower()
                   .replace("&","and")
                   .replace("/","_")
                   .replace(" ","_")
                   .replace("-", "_")
            )
            score = info.get("score", None)
            if score is not None:
                by_person[speaker][f"ann_{key}_count"] += 1
                by_person[speaker][f"ann_{key}_sum_score"] += float(score)

    # return dict of dicts
    return {p: dict(cnt) for p, cnt in by_person.items()}

def load_conference(conf_path: Path):
    conf_name = conf_path.name  # e.g., 2021MZT
    year = int(conf_name[:4])
    conference = conf_name[4:]

    session_outcomes = _load_json(conf_path / f"{conf_name}_session_outcomes.json")

    # features_*.json (session-level context; we’ll prefix ctx_*)
    features = {}
    for fp in conf_path.glob("features_*.json"):
        sid = fp.stem.replace("features_", "")
        features[sid] = _load_json(fp)

    return year, conference, session_outcomes, features

# ========= BUILD PERSON-SESSION =========
all_ps = []
for conf_path in sorted(DATA_DIR.iterdir()):
    if not conf_path.is_dir():
        continue
    conf_name = conf_path.name
    core = conf_path / f"{conf_name}_session_outcomes.json"
    if not core.exists():
        continue

    year, conference, session_outcomes, features = load_conference(conf_path)
    special = {"missing_names", "people_not_in_any_team"}
    sessions = [sid for sid in session_outcomes.keys() if sid not in special]

    for sid in sessions:
        so = session_outcomes[sid]
        facilitators = set(so.get("facilitators", []) or [])
        speakers     = set(so.get("all_speakers", []) or [])

        # team members from teams{}
        members = set()
        for _, tinfo in (so.get("teams", {}) or {}).items():
            for m in tinfo.get("members", []) or []:
                members.add(m)

        people = sorted(facilitators | speakers | members)
        ctx = {f"ctx_{k}": v for k, v in (features.get(sid, {}) or {}).items()}
        pmet = extract_person_metrics_from_session_data(conf_path, sid)

        for person in people:
            if person in facilitators:
                role_in_session = "facilitator"
            elif person in members:
                role_in_session = "member"
            elif person in speakers:
                role_in_session = "participant"
            else:
                role_in_session = "unknown"

            row = {
                "person_name": person,
                "conference": conference,
                "year": year,
                "session_id": sid,
                "role_in_session": role_in_session,
                **ctx
            }
            row.update(pmet.get(person, {}))
            all_ps.append(row)

ps_df = pd.DataFrame(all_ps)

# write person-session (audit)
if WRITE_PERSON_SESSION and not ps_df.empty:
    ps_out = OUTPUT_DIR / "ALL_person_session.csv"
    ps_df.to_csv(ps_out, index=False)
    print(f"Wrote person-session audit: {ps_out}  ({len(ps_df)} rows)")

# ========= AGGREGATE → PERSON-YEAR (SUM COUNTS; WEIGHTED MEANS FOR ann_*_mean_score) =========
if ps_df.empty:
    raise SystemExit("No person-session rows. Check DATA_DIR structure/files.")

keys = ["person_name","conference","year"]

# identify columns
p_cols = [c for c in ps_df.columns if c.startswith("p_")]
ctx_cols = [c for c in ps_df.columns if c.startswith("ctx_")]
ann_count_cols = [c for c in ps_df.columns if c.startswith("ann_") and c.endswith("_count")]
ann_sum_cols   = [c for c in ps_df.columns if c.startswith("ann_") and c.endswith("_sum_score")]

# base aggregations
agg_map = {}
# p_* sum (counts/segments) and duration sum
for c in p_cols:
    agg_map[c] = "sum"
# ctx_* mean across sessions
for c in ctx_cols:
    agg_map[c] = "mean"
# annotations: sum counts and sum_scores
for c in ann_count_cols + ann_sum_cols:
    agg_map[c] = "sum"

py_base = ps_df.groupby(keys, as_index=False).agg(agg_map)

# add sessions_total (unique sessions attended)
sesh_counts = (
    ps_df.groupby(keys, as_index=False)["session_id"]
         .nunique()
         .rename(columns={"session_id":"sessions_total"})
)
py = py_base.merge(sesh_counts, on=keys, how="left")

# derive role tallies per person-year from person-session
role_tallies = (
    ps_df.assign(one=1)
         .pivot_table(index=keys, columns="role_in_session", values="one", aggfunc="sum", fill_value=0)
         .reset_index()
         .rename(columns={"facilitator":"facilitator", "member":"member", "participant":"participant", "unknown":"unknown"})
)
# ensure all role columns exist
for col in ["facilitator","member","participant","unknown"]:
    if col not in role_tallies.columns:
        role_tallies[col] = 0
py = py.merge(role_tallies, on=keys, how="left")

# weighted means for each annotation mean_score = sum_score / count
ann_categories = set([c.replace("ann_","").replace("_count","") for c in ann_count_cols])
for cat in sorted(ann_categories):
    cnt_col = f"ann_{cat}_count"
    sum_col = f"ann_{cat}_sum_score"
    mean_col = f"ann_{cat}_mean_score"
    if cnt_col in py.columns and sum_col in py.columns:
        py[mean_col] = np.where(py[cnt_col].fillna(0) > 0,
                                pd.to_numeric(py[sum_col], errors="coerce") / pd.to_numeric(py[cnt_col], errors="coerce"),
                                np.nan)

# ===== dtype enforcement: ints for counts; floats for durations/scores =====
# counts (discrete)
count_like = [
    "p_turns","p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
    "sessions_total","facilitator","member","participant","unknown"
] + ann_count_cols

for c in count_like:
    if c in py.columns:
        py[c] = pd.to_numeric(py[c], errors="coerce").fillna(0).round().astype(int)

# sums of scores are numeric; they’ll typically be floats but are sums of discrete scores (keep as int if integral)
for c in ann_sum_cols:
    if c in py.columns:
        s = pd.to_numeric(py[c], errors="coerce").fillna(0)
        py[c] = np.where(np.isclose(s, np.round(s)), s.round().astype(int), s)  # keep int if whole, else float

# speaking duration is continuous; keep float (round for readability)
if "p_speaking_duration_sec" in py.columns:
    py["p_speaking_duration_sec"] = pd.to_numeric(py["p_speaking_duration_sec"], errors="coerce").round(3)

# ===== OPTIONAL: derive role flags =====
py["role_primary"] = py[["facilitator","member","participant","unknown"]].idxmax(axis=1)
py["role_facilitator"]   = (py["facilitator"]   > 0).astype(int)
py["role_member"]        = (py["member"]        > 0).astype(int)
py["role_participant"]   = (py["participant"]   > 0).astype(int)
py["role_nonfacilitator"]= (py["role_facilitator"] == 0).astype(int)

# If you have team files to compute on_team / funded, merge here (skipped because not loaded in this block)

# write person-year
py_out = OUTPUT_DIR / "ALL_person_year_FIXED.csv"
py.sort_values(["person_name","year","conference"]).to_csv(py_out, index=False)
print(f"Wrote person-year: {py_out}  ({len(py)} rows)")

# ========= AGGREGATE → PERSON (across all years) =========
sum_cols_person = [
    "p_turns","p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
    "sessions_total","facilitator","member","participant","unknown"
] + ann_count_cols + ann_sum_cols

mean_cols_person = [c for c in py.columns if c.startswith(("ctx_"))]  # context means (optional)
mean_cols_person += [c for c in py.columns if c.endswith("_mean_score")]  # annotation means

agg_person = {}
for c in sum_cols_person:
    if c in py.columns: agg_person[c] = "sum"
for c in mean_cols_person:
    if c in py.columns: agg_person[c] = "mean"

person = py.groupby("person_name", as_index=False).agg(agg_person)

# enforce ints again for counts
for c in sum_cols_person:
    if c in person.columns:
        person[c] = pd.to_numeric(person[c], errors="coerce").fillna(0).round().astype(int)

# keep durations and mean scores as floats (rounded)
for c in [x for x in person.columns if x.endswith("_mean_score") or x.startswith(("ctx_","p_speaking_duration_sec"))]:
    person[c] = pd.to_numeric(person[c], errors="coerce").round(3)

person_out = OUTPUT_DIR / "all_data_df-agg-ppl_FIXED.csv"
person.sort_values(["person_name"]).to_csv(person_out, index=False)
print(f"Wrote person-level: {person_out}  ({len(person)} rows)")

# ========= QUICK SANITY: show any columns that are float but expected int ========
def check_int_columns(df, cols, label):
    bad = []
    for c in cols:
        if c in df.columns and not pd.api.types.is_integer_dtype(df[c]):
            bad.append(c)
    if bad:
        print(f"[WARN] {label} columns not integer as expected:", bad)
    else:
        print(f"[OK] All {label} columns are integer.")

int_cols_py = ["p_turns","p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
               "sessions_total","facilitator","member","participant","unknown"] + ann_count_cols
check_int_columns(py, int_cols_py, "person-year count-like")

int_cols_person = [c for c in int_cols_py + ann_sum_cols if c in person.columns]
check_int_columns(person, int_cols_person, "person-level count-like")

Wrote person-session audit: /Users/maxchalekson/Desktop/outputs/ALL_person_session.csv  (2073 rows)
Wrote person-year: /Users/maxchalekson/Desktop/outputs/ALL_person_year_FIXED.csv  (790 rows)
Wrote person-level: /Users/maxchalekson/Desktop/outputs/all_data_df-agg-ppl_FIXED.csv  (670 rows)
[OK] All person-year count-like columns are integer.
[OK] All person-level count-like columns are integer.


### fixing the names

In [7]:
import re, unicodedata
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

# ========= PATHS =========
IN_DIR = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/")
PS_IN  = IN_DIR / "ALL_person_session.csv"
PY_IN  = IN_DIR / "ALL_person_year_FIXED.csv"
P_IN   = IN_DIR / "all_data_df-agg-ppl_FIXED.csv"

OUT_DIR = IN_DIR
MAP_OUT = OUT_DIR / "name_clean_map.csv"
PS_OUT  = OUT_DIR / "ALL_person_session_CLEANED.csv"
PY_OUT  = OUT_DIR / "ALL_person_year_FIXED_CLEANED.csv"
P_OUT   = OUT_DIR / "all_data_df-agg-ppl_FIXED_CLEANED.csv"

# ========= LOAD =========
ps = pd.read_csv(PS_IN)
py = pd.read_csv(PY_IN)
p  = pd.read_csv(P_IN)

# ========= NAME NORMALIZATION =========
DEGREE_SUFFIXES = [
    r"\bph\.?d\.?\b", r"\bmd\b", r"\bmd-ph\.?d\.?\b", r"\bscd\b", r"\bmsc\b", r"\bms\b",
    r"\bba\b", r"\bma\b", r"\bmba\b", r"\bdvm\b", r"\bdds\b"
]
ORG_DELIMS = [r"\s*-\s*", r"\s*—\s*", r"\s*–\s*", r"\s*\|\s*", r"\s*@\s*"]
TRAILERS   = [r"\(.*?\)", r"\[.*?\]", r"\{.*?\}"]

# If you know any manual fixes, put them here (raw or cleaned key -> canonical value)
alias_map = {
    # "Max Chalekson - UCLA": "Max Chalekson",
    # "Evey Huang, PhD": "Evey Huang",
}

def strip_accents(text: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))

def basic_name_clean(s: str) -> str:
    if not isinstance(s, str) or not s.strip():
        return ""
    x = s.strip()

    # remove emails
    x = re.sub(r"\b\S+@\S+\.\S+\b", " ", x)
    # drop anything in (), [], {}
    for patt in TRAILERS:
        x = re.sub(patt, " ", x)
    # drop degree suffixes
    for deg in DEGREE_SUFFIXES:
        x = re.sub(deg, " ", x, flags=re.IGNORECASE)
    # split off affiliation/org
    for delim in ORG_DELIMS:
        x = re.split(delim, x)[0]

    # remove titles
    x = re.sub(r"^\s*(dr|prof|mr|ms|mrs)\.?\s+", " ", x, flags=re.IGNORECASE)

    # tidy punctuation/whitespace
    x = re.sub(r"[^\w\s'.-]", " ", x)            # keep letters/digits/space/apostrophe/dot/hyphen
    x = re.sub(r"\s+", " ", x).strip()

    # accents -> ascii, title-case words
    x = strip_accents(x)
    x = " ".join(w.capitalize() for w in x.split())

    # length guard
    return x[:120].strip()

def final_clean(raw: str) -> str:
    if not isinstance(raw, str): return ""
    if raw in alias_map: return alias_map[raw]
    cleaned = basic_name_clean(raw)
    return alias_map.get(cleaned, cleaned)

# Build mapping from the *rawest* table (person-session)
raw_names = ps["person_name"].astype(str).fillna("")
freq = Counter(raw_names)
unique_raw = pd.Series(sorted(set(raw_names)))
cleaned = unique_raw.apply(final_clean)

def keyize(s: str) -> str:
    s2 = s.lower()
    s2 = re.sub(r"[^a-z]", "", s2)  # letters only for a coarse cluster key
    return s2

map_df = pd.DataFrame({
    "raw_name": unique_raw,
    "clean_name": cleaned,
})
map_df["raw_freq"] = map_df["raw_name"].map(freq)
map_df["cluster_key"] = map_df["clean_name"].apply(keyize)

# choose canonical per cluster (most frequent raw -> its cleaned)
canon_by_key = {}
for key, grp in map_df.groupby("cluster_key"):
    idx = grp["raw_freq"].idxmax()
    canon_by_key[key] = grp.loc[idx, "clean_name"]
map_df["canonical_name"] = map_df["cluster_key"].map(canon_by_key)

# Save mapping for review
map_df.sort_values(["cluster_key","clean_name","raw_freq"], ascending=[True, True, False]).to_csv(MAP_OUT, index=False)
print(f"[OK] Wrote name mapping: {MAP_OUT}")

# Helper to apply mapping (raw->canonical) with fallback to cleaner
raw_to_canon = dict(zip(map_df["raw_name"], map_df["canonical_name"]))

def apply_canonical(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["person_name"] = df["person_name"].map(raw_to_canon).fillna(df["person_name"].apply(final_clean))
    return df

# ========= APPLY TO PERSON-SESSION (and write) =========
ps_clean = apply_canonical(ps)
ps_clean.to_csv(PS_OUT, index=False)
print(f"[OK] Wrote cleaned person-session: {PS_OUT}  (rows={len(ps_clean)})")

# ========= RE-AGG LOGIC (keeps ints for counts, floats for means) =========
def build_agg_dict(df: pd.DataFrame):
    sum_cols = []
    mean_cols = []
    for c in df.columns:
        if c in ["person_name","conference","year","session_id","role_in_session","role_primary","team_ids"]:
            continue
        # counts / sums (discrete events) -> SUM
        if (c.endswith("_count") or c.endswith("_sum_score") or
            c in ["p_turns","p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
                  "sessions_total","facilitator","member","participant","unknown",
                  "teams_funded","teams_unfunded","teams_total",
                  "role_facilitator","role_nonfacilitator","role_on_team","role_in_funded",
                  "role_member","role_participant"]):
            sum_cols.append(c)
        # means / continuous -> MEAN
        elif (c.endswith("_mean_score") or c.endswith("_mean_score_weighted") or
              c.startswith("ctx_") or c in ["p_speaking_duration_sec"]):
            mean_cols.append(c)
        else:
            # default: if numeric and not a known count, treat as mean to be safe
            if pd.api.types.is_numeric_dtype(df[c]):
                mean_cols.append(c)
    agg = {**{c:"sum" for c in sum_cols}, **{c:"mean" for c in mean_cols}}
    return agg, sum_cols

def cast_int_counts(df: pd.DataFrame, count_cols):
    for c in count_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).round().astype(int)
    return df

# ========= RE-AGG PERSON–YEAR =========
py_clean = apply_canonical(py)
agg_py, sum_cols_py = build_agg_dict(py_clean)
py_final = (py_clean
            .groupby(["person_name","conference","year"], as_index=False)
            .agg(agg_py)
           )
py_final = cast_int_counts(py_final, sum_cols_py + ["year"])
py_final.to_csv(PY_OUT, index=False)
print(f"[OK] Wrote cleaned person–year: {PY_OUT}  (rows={len(py_final)})")

# ========= RE-AGG PERSON-LEVEL =========
p_clean = apply_canonical(p)
agg_p, sum_cols_p = build_agg_dict(p_clean)
p_final = (p_clean
           .groupby(["person_name"], as_index=False)
           .agg(agg_p)
          )
p_final = cast_int_counts(p_final, sum_cols_p)
p_final.to_csv(P_OUT, index=False)
print(f"[OK] Wrote cleaned person-level: {P_OUT}  (rows={len(p_final)})")

# ========= QUICK DIAGNOSTICS =========
sus = map_df[ (map_df["clean_name"].eq("")) | (~map_df["clean_name"].str.contains(r"\s")) ]
if not sus.empty:
    print("\n[Heads-up] Some names look empty or single-token; consider alias_map fixes. Examples:")
    print(sus.sort_values("raw_freq", ascending=False).head(15)[["raw_name","clean_name","canonical_name","raw_freq"]].to_string(index=False))
else:
    print("\n[OK] No obviously empty/single-token names in mapping.")

[OK] Wrote name mapping: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/name_clean_map.csv
[OK] Wrote cleaned person-session: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_session_CLEANED.csv  (rows=2073)
[OK] Wrote cleaned person–year: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_year_FIXED_CLEANED.csv  (rows=771)
[OK] Wrote cleaned person-level: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/all_data_df-agg-ppl_FIXED_CLEANED.csv  (rows=637)

[Heads-up] Some names look empty or single-token; consider alias_map fixes. Examples:
                raw_name clean_name canonical_name  raw_freq
    Marie-Claire Arrieta      Marie          Marie         4
            John-Paul Yu       John           John         4
             Gang-yu Liu       Gang           Gang         4
   Anna-Karin Gustavsson       An

## rebuilding the analysis part 

In [11]:
# =========================================
# REBUILD ANALYSIS: Charts + Regressions + ROC (robust to collinearity)
# Requires: pandas, numpy, matplotlib, statsmodels, scikit-learn, xlsxwriter, tabulate
# =========================================

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.sm_exceptions import PerfectSeparationError
from patsy import dmatrices
import numpy.linalg as npl

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, classification_report
from sklearn.impute import SimpleImputer

# ------------------ CONFIG ------------------
DATA_CSV = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_year.csv")
OUT_DIR  = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_rebuild")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS     = 5
INCLUDE_CTX  = True   # include ctx_* features along with p_* and ann_* in regressions and classifiers

# ------------------ LOAD ------------------
df = pd.read_csv(DATA_CSV)

# ---- Ensure role flags exist (derive if missing) ----
def ensure_role_flags(df):
    df = df.copy()

    # role_facilitator
    if "role_facilitator" not in df.columns:
        if "facilitator" in df.columns:
            df["role_facilitator"] = (pd.to_numeric(df["facilitator"], errors="coerce").fillna(0) > 0).astype(int)
        elif "role_primary" in df.columns:
            df["role_facilitator"] = (df["role_primary"].astype(str).str.lower() == "facilitator").astype(int)
        else:
            df["role_facilitator"] = 0

    # role_on_team
    if "role_on_team" not in df.columns:
        if "teams_total" in df.columns:
            df["role_on_team"] = (pd.to_numeric(df["teams_total"], errors="coerce").fillna(0) > 0).astype(int)
        elif "member" in df.columns:
            df["role_on_team"] = (pd.to_numeric(df["member"], errors="coerce").fillna(0) > 0).astype(int)
        else:
            df["role_on_team"] = 0

    # role_in_funded
    if "role_in_funded" not in df.columns:
        if "teams_funded" in df.columns:
            df["role_in_funded"] = (pd.to_numeric(df["teams_funded"], errors="coerce").fillna(0) > 0).astype(int)
        else:
            df["role_in_funded"] = 0

    for c in ["role_facilitator","role_on_team","role_in_funded"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)
    return df

df = ensure_role_flags(df)

# Which flags actually exist?
role_flags_available = [c for c in ["role_facilitator","role_on_team","role_in_funded"] if c in df.columns]

# ------------------ FEATURE SETS ------------------
# Person-level behavioral (transcript-derived)
p_feats   = [c for c in df.columns if c.startswith("p_") and pd.api.types.is_numeric_dtype(df[c])]
# Annotation categories
ann_feats = [c for c in df.columns if c.startswith("ann_") and pd.api.types.is_numeric_dtype(df[c])]
# Session-level context (if present)
ctx_feats = [c for c in df.columns if c.startswith("ctx_") and pd.api.types.is_numeric_dtype(df[c])]
if not INCLUDE_CTX:
    ctx_feats = []

num_feats = sorted(list(set(p_feats + ann_feats + ctx_feats)))

# Drop zero-variance / all-missing features
good_feats = []
for c in num_feats:
    s = pd.to_numeric(df[c], errors="coerce")
    if s.notna().any() and s.std(skipna=True) > 0:
        good_feats.append(c)
num_feats = sorted(good_feats)

print(f"Using {len(num_feats)} numeric features.")

# ------------------ CHARTS: Helper Plots ------------------
def bar_mean_with_ci(data, by_flag, metric, title, out_png):
    tmp = data[[by_flag, metric]].dropna()
    tmp[by_flag] = tmp[by_flag].astype(int)
    groups = tmp.groupby(by_flag)[metric]
    means = groups.mean()
    ns    = groups.size()
    sds   = groups.std()
    cis = 1.96 * sds / np.sqrt(ns.clip(lower=1))  # 95% CI

    fig, ax = plt.subplots(figsize=(5,4))
    ax.bar(["No","Yes"], [means.get(0, np.nan), means.get(1, np.nan)])
    ax.errorbar([0,1], [means.get(0, np.nan), means.get(1, np.nan)],
                yerr=[cis.get(0,0), cis.get(1,0)], fmt='none', capsize=4)
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.set_xlabel(by_flag)
    plt.tight_layout()
    fig.savefig(out_png, dpi=180)
    plt.close(fig)

def box_by_flag(data, by_flag, metric, title, out_png):
    tmp = data[[by_flag, metric]].dropna()
    tmp[by_flag] = tmp[by_flag].astype(int)
    fig, ax = plt.subplots(figsize=(5,4))
    ax.boxplot([tmp.loc[tmp[by_flag]==0, metric], tmp.loc[tmp[by_flag]==1, metric]],
               labels=["No","Yes"], showfliers=False)
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.set_xlabel(by_flag)
    plt.tight_layout()
    fig.savefig(out_png, dpi=180)
    plt.close(fig)

# Pick a concise subset of metrics to visualize (add more if you like)
bar_metrics = [
    "p_interruptions_made","p_overlaps","p_screenshare_segments","p_nods_received",
    "ann_knowledge_sharing_count","ann_evaluation_practices_count",
    "ann_coordination_and_decision_practices_count","ann_relational_climate_count",
    "ann_participation_dynamics_count"
]
box_metrics = ["p_speaking_duration_sec","p_turns"]

# Make charts for each contrast flag that exists
for flag in role_flags_available:
    for m in bar_metrics:
        if m in df.columns:
            fn = OUT_DIR / f"bar_{m}_{flag}.png"
            bar_mean_with_ci(df, flag, m, f"{m} by {flag}", fn)
    for m in box_metrics:
        if m in df.columns:
            fn = OUT_DIR / f"box_{m}_{flag}.png"
            box_by_flag(df, flag, m, f"{m} by {flag}", fn)

print("[OK] Wrote bar/box charts.")

# ------------------ REGRESSIONS (Logit with FE, full-rank safe) ------------------
def _drop_problem_cols(X_df):
    """Drop zero-variance, duplicate columns; if still rank-deficient, keep a full-rank subset via QR."""
    X = X_df.copy()

    # Drop all-constant / zero-variance
    keep = [c for c in X.columns if X[c].std(ddof=0) > 0]
    X = X[keep]

    # Drop exact-duplicate columns
    seen = {}
    dedup_keep = []
    for c in X.columns:
        key = tuple(np.asarray(X[c]).round(12))
        if key in seen:
            continue
        seen[key] = c
        dedup_keep.append(c)
    X = X[dedup_keep]

    # Full-rank via QR pivot if needed
    A = X.values
    rank = np.linalg.matrix_rank(A)
    if rank == A.shape[1]:
        return X

    Q, R, piv = npl.qr(A, mode='reduced', pivoting=True)
    indep_idx = piv[:rank]
    keep_cols = [X.columns[i] for i in indep_idx]
    return X[keep_cols]

def logit_with_fe(data, target, predictors, fe_conf=True, fe_year=True, add_sess=True):
    """
    Build design via patsy, drop collinear columns, fit Logit.
    If MLE fails (singular/complete separation), fall back to fit_regularized.
    Returns: (data_used, model_or_result, robust_or_none, formula_string)
    """
    # Build formula
    rhs = []
    if add_sess and "sessions_total" in data.columns:
        rhs.append("sessions_total")
    rhs += [f"`{x}`" if (" " in x or "-" in x) else x for x in predictors]
    if fe_conf and "conference" in data.columns:
        rhs.append("C(conference)")
    if fe_year and "year" in data.columns:
        rhs.append("C(year)")
    formula = f"{target} ~ " + " + ".join(rhs)

    # Design matrices
    try:
        y, X = dmatrices(formula, data=data, return_type="dataframe", NA_action="drop")
    except Exception as e:
        print(f"[SKIP] Patsy failed for {target}: {e}")
        return None, None, None, formula

    y_vec = np.asarray(y).ravel().astype(int)
    if np.unique(y_vec).size < 2:
        print(f"[SKIP] {target}: only one class after NA drop.")
        return None, None, None, formula

    # Hold intercept, clean the rest
    const = None
    if "Intercept" in X.columns:
        const = X["Intercept"].copy()
        X = X.drop(columns=["Intercept"])
    X = _drop_problem_cols(X)
    if const is not None:
        X.insert(0, "Intercept", const.values)

    # Fit (request robust SEs directly); fallback to regularized if needed
    try:
        model = sm.Logit(y_vec, X)
        result = model.fit(disp=0, cov_type="HC3")   # <-- robust SEs here
        robust = result                               # already robust
        used = pd.concat([y.reset_index(drop=True), X.reset_index(drop=True)], axis=1)
        return used, result, robust, formula
    except (np.linalg.LinAlgError, PerfectSeparationError) as e:
        print(f"[Info] MLE failed for {target} ({type(e).__name__}); trying regularized fit.")
        model = sm.Logit(y_vec, X)
        result = model.fit_regularized(alpha=1.0, L1_wt=0.5, disp=0)
        robust = None  # robust SEs not available here
        used = pd.concat([y.reset_index(drop=True), X.reset_index(drop=True)], axis=1)
        return used, result, robust, formula

def tidy_from_result(result, robust_or_none, model_label):
    params = (robust_or_none.params if robust_or_none is not None else result.params)
    bse    = (robust_or_none.bse    if robust_or_none is not None else result.bse)
    pvals  = (robust_or_none.pvalues if robust_or_none is not None else result.pvalues)
    conf   = (robust_or_none.conf_int() if robust_or_none is not None else result.conf_int())

    out = pd.DataFrame({
        "predictor": params.index,
        "coef": params.values,
        "std_err": bse.values,
        "p_value": pvals.values,
        "ci_low": conf[0].values,
        "ci_high": conf[1].values,
        "model": model_label,
        "n_obs": int(result.nobs),
        "llf": result.llf,
        "llnull": getattr(result, "llnull", np.nan),
        "pseudo_r2": (1 - (result.llf / result.llnull)) if getattr(result, "llnull", 0) not in (0, np.nan) else np.nan,
        "llr_pvalue": getattr(result, "llr_pvalue", np.nan)
    })
    return out

tidy_list = []
summaries = []

for tgt, desc in [
    ("role_facilitator","Facilitator vs Non-facilitator"),
    ("role_on_team","On-team vs Not-on-team"),
    ("role_in_funded","Funded vs Not-funded")
]:
    if tgt not in role_flags_available:
        continue
    used, res, rob, form = logit_with_fe(df, tgt, predictors=num_feats, fe_conf=True, fe_year=True, add_sess=True)
    if res is None:
        continue

    # summary text
    summ = [f"=== {desc} ({tgt}) ===", f"Formula: {form}", res.summary2().as_text()]
    if rob is None:
        summ.append("\nNote: Regularized fit used (robust SEs not available).")
    else:
        summ.append("\nRobust (HC3) SEs used in tidy table.")
    summaries.append("\n".join(summ))

    tidy_list.append(tidy_from_result(res, rob, desc))

# Save tidy combined and summaries
if tidy_list:
    tidy_df = pd.concat(tidy_list, ignore_index=True)
    tidy_csv = OUT_DIR / "regression_tidy_results.csv"
    tidy_df.to_csv(tidy_csv, index=False)

    with open(OUT_DIR / "regression_model_summaries.txt","w") as f:
        f.write("\n\n".join(summaries))

    # Nicely formatted per-model tables
    def tidy_to_display(df_tidy, model_name):
        sub = df_tidy[df_tidy["model"]==model_name].copy()
        sub.loc[sub["predictor"].str.startswith("C(conference)"), "predictor"] = sub["predictor"].str.replace("C(conference)","Conf_FE", regex=False)
        sub.loc[sub["predictor"].str.startswith("C(year)"), "predictor"] = sub["predictor"].str.replace("C(year)","Year_FE", regex=False)
        sub["Coef (SE)"] = sub["coef"].round(3).astype(str) + " (" + sub["std_err"].round(3).astype(str) + ")"
        sub["p"] = sub["p_value"].apply(lambda x: f"{x:.3g}")
        keep = ["predictor","Coef (SE)","ci_low","ci_high","p","n_obs","pseudo_r2","llr_pvalue"]
        sub = sub[keep].rename(columns={"predictor":"Term","ci_low":"CI 2.5%","ci_high":"CI 97.5%","p":"p-value","n_obs":"N","pseudo_r2":"Pseudo R2","llr_pvalue":"LLR p"})
        return sub

    tables = {}
    for model_name in tidy_df["model"].unique():
        disp = tidy_to_display(tidy_df, model_name)
        safe = "".join(ch for ch in model_name if ch.isalnum() or ch in "_- ").strip().replace(" ","_")
        disp.to_csv(OUT_DIR / f"reg_table_{safe}.csv", index=False)
        try:
            disp.to_markdown(OUT_DIR / f"reg_table_{safe}.md", index=False)
        except Exception:
            pass
        tables[model_name] = disp

    xlsx_path = OUT_DIR / "regression_tables_by_model.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as xw:
        for model_name, disp in tables.items():
            sheet = model_name[:31]
            disp.to_excel(xw, index=False, sheet_name=sheet)

    print(f"[OK] Regressions saved: {tidy_csv} and {xlsx_path}")
else:
    print("[SKIP] No regression models were fit (missing targets or single-class after filtering).")

# ------------------ MULTICOLLINEARITY (VIF on key behaviors) ------------------
vif_feats = [c for c in ["p_speaking_duration_sec","p_turns","p_interruptions_made","p_overlaps",
                         "p_screenshare_segments","p_smile_self_total","p_smile_other_total",
                         "p_nods_received"] if c in df.columns]
if vif_feats:
    X_vif = df[vif_feats].fillna(0)
    X_vif = sm.add_constant(X_vif)
    vif_table = pd.DataFrame({
        "feature": X_vif.columns,
        "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    })
    vif_table.to_csv(OUT_DIR / "vif_behaviors.csv", index=False)
    print("[OK] VIF table written.")
else:
    print("[SKIP] No VIF — none of the key behavior features found.")

# ------------------ PREDICTIVE: ROC Curves with behavior-only ------------------
targets = [(t, d) for (t, d) in [
    ("role_facilitator","Facilitator vs Non-facilitator"),
    ("role_on_team","On-team vs Not-on-team"),
    ("role_in_funded","Funded vs Not-funded")
] if t in role_flags_available]

X_cols = num_feats  # p_, ann_, and (ctx_ if INCLUDE_CTX)
if targets and X_cols:
    numeric_transform = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("scale", StandardScaler())
    ])
    preprocess = ColumnTransformer(transformers=[("num", numeric_transform, X_cols)], remainder="drop")

    logit = LogisticRegression(
        penalty="l2",
        solver="liblinear",
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE
    )
    clf = Pipeline(steps=[("prep", preprocess), ("clf", logit)])
    cv  = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    summary_rows = []
    for target, desc in targets:
        dsub = df.dropna(subset=[target]).copy()
        if dsub[target].nunique() < 2:
            print(f"[SKIP] ROC for {target}: single class present.")
            continue

        y = dsub[target].astype(int).values
        X = dsub[X_cols]

        # CV probs + ROC
        y_prob = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:,1]
        fpr, tpr, _ = roc_curve(y, y_prob)
        roc_auc = auc(fpr, tpr)

        # Classification report at 0.5
        y_hat = (y_prob >= 0.5).astype(int)
        cr = classification_report(y, y_hat, output_dict=True, zero_division=0)
        pd.DataFrame(cr).to_csv(OUT_DIR / f"clf_classification_report_{target}.csv")

        # CV metric summary
        scoring = {"roc_auc": "roc_auc","accuracy":"accuracy","precision":"precision","recall":"recall","f1":"f1"}
        cv_res = cross_validate(clf, X, y, cv=cv, scoring=scoring, return_train_score=False, n_jobs=-1)
        row = {"target": target, "description": desc}
        for m in scoring:
            row[f"{m}_mean"] = float(cv_res[f"test_{m}"].mean())
            row[f"{m}_sd"]   = float(cv_res[f"test_{m}"].std())
        summary_rows.append(row)

        # ROC plot
        fig, ax = plt.subplots(figsize=(5,4))
        ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
        ax.plot([0,1],[0,1], linestyle="--")
        ax.set_title(f"ROC — {desc}")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.legend(loc="lower right")
        plt.tight_layout()
        fig.savefig(OUT_DIR / f"clf_roc_{target}.png", dpi=180)
        plt.close(fig)

    if summary_rows:
        summary_df = pd.DataFrame(summary_rows).sort_values("target")
        summary_df.to_csv(OUT_DIR / "clf_results_summary.csv", index=False)
        print("[OK] Predictive ROC + summaries written.")
else:
    print("[SKIP] ROC — no targets available or no numeric features.")

print("\nOutputs written to:", OUT_DIR)

Using 33 numeric features.


/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_68986/346889039.py:124: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([tmp.loc[tmp[by_flag]==0, metric], tmp.loc[tmp[by_flag]==1, metric]],
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_68986/346889039.py:124: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([tmp.loc[tmp[by_flag]==0, metric], tmp.loc[tmp[by_flag]==1, metric]],
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_68986/346889039.py:124: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([tmp.loc[tmp[by_flag]==0, metric], tmp.loc[tmp[by_flag]==1, metri

[OK] Wrote bar/box charts.
[Info] MLE failed for role_facilitator (LinAlgError); trying regularized fit.
[SKIP] role_in_funded: only one class after NA drop.
[OK] Regressions saved: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_rebuild/regression_tidy_results.csv and /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_rebuild/regression_tables_by_model.xlsx
[OK] VIF table written.
[SKIP] ROC for role_in_funded: single class present.
[OK] Predictive ROC + summaries written.

Outputs written to: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_rebuild


## analysis - 3 categorical

In [18]:
# =========================================
# Meeting-level analysis (person-year data)
# - Outlier checks for controls
# - Single-feature OLS (HC3 SE) on num_teams / num_funded_teams
# - Tidy CSVs + Excel
# - Coef barplots with 95% CI
# - Big grid image: hist + scatters for each feature
# =========================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ------------------ CONFIG ------------------
DATA_CSV = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_year_WITH_OUTCOMES.csv")
OUT_DIR  = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Which outcomes to model (must exist in the CSV)
OUTCOMES = ["num_teams", "num_funded_teams"]

# Which controls to include in every single-feature regression
CONTROLS = ["p_speaking_duration_sec", "p_turns"]  # (Evey’s suggestion)

# Number of bars to show (by smallest p) in coef barplots
TOP_N_FOR_PLOT = 15

RANDOM_STATE = 42

# ------------------ LOAD ------------------
df = pd.read_csv(DATA_CSV)

# Basic hygiene: make sure outcomes exist & are numeric
for c in OUTCOMES:
    if c not in df.columns:
        raise ValueError(f"Outcome '{c}' not found in {DATA_CSV}.")
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Predictors = p_* and ann_* numeric columns
p_feats   = [c for c in df.columns if c.startswith("p_")   and pd.api.types.is_numeric_dtype(df[c])]
ann_feats = [c for c in df.columns if c.startswith("ann_") and pd.api.types.is_numeric_dtype(df[c])]
predictors = sorted(list(set(p_feats + ann_feats)))

# Drop outcomes / predictors that are entirely missing
df = df.copy()
df[predictors + OUTCOMES] = df[predictors + OUTCOMES].apply(pd.to_numeric, errors="coerce")

# ------------------ Quick outlier checks for controls ------------------
def quick_outlier_plots(data: pd.DataFrame, cols: list, out_dir: Path):
    for c in cols:
        if c not in data.columns:
            continue
        s = pd.to_numeric(data[c], errors="coerce")
        fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
        # Histogram (log x if long tail)
        ax[0].hist(s.dropna(), bins=40)
        ax[0].set_title(f"{c} — histogram")
        ax[0].set_xlabel(c); ax[0].set_ylabel("count")
        # Boxplot (shows extreme points)
        ax[1].boxplot(s.dropna(), vert=True, showfliers=True)
        ax[1].set_title(f"{c} — boxplot")
        ax[1].set_ylabel(c)
        plt.tight_layout()
        fig.savefig(out_dir / f"outlier_{c}.png", dpi=180)
        plt.close(fig)

# scatter against outcomes (to see leverage)
def scatter_vs_outcomes(data: pd.DataFrame, cols: list, outcomes: list, out_dir: Path):
    for c in cols:
        if c not in data.columns:
            continue
        for y in outcomes:
            sub = data[[c, y]].dropna()
            if sub.empty: 
                continue
            fig, ax = plt.subplots(figsize=(4.8, 3.6))
            ax.scatter(sub[c], sub[y], s=10, alpha=0.6)
            ax.set_xlabel(c); ax.set_ylabel(y)
            ax.set_title(f"{y} vs {c}")
            plt.tight_layout()
            fig.savefig(out_dir / f"scatter_{y}_vs_{c}.png", dpi=180)
            plt.close(fig)

quick_outlier_plots(df, CONTROLS, OUT_DIR)
scatter_vs_outcomes(df, CONTROLS, OUTCOMES, OUT_DIR)

# ------------------ Utility: single-feature OLS with HC3 robust SE ------------------
def run_single_feature_ols(data: pd.DataFrame, outcome: str, feature: str, controls: list):
    cols_base = [outcome, feature]
    # exclude the focal feature from the controls to avoid duplicates
    controls_eff = [c for c in controls if (c in data.columns and c != feature)]
    cols = cols_base + controls_eff

    d = data[cols].dropna()
    if d[outcome].nunique() <= 1:
        return None  # cannot regress on constant outcome

    y = d[outcome].astype(float).values
    X = d[[feature] + controls_eff].astype(float)
    X = sm.add_constant(X, has_constant="add")

    model = sm.OLS(y, X).fit(cov_type="HC3")  # HC3 robust SE

    # now `feature` is unique in X, so these are scalars
    out = {
        "outcome": outcome,
        "feature": feature,
        "coef": float(model.params[feature]),
        "std_err": float(model.bse[feature]),
        "t": float(model.tvalues[feature]),
        "p_value": float(model.pvalues[feature]),
        "ci_low": float(model.conf_int().loc[feature, 0]),
        "ci_high": float(model.conf_int().loc[feature, 1]),
        "N": int(model.nobs),
        "R2": float(model.rsquared),
        "Adj_R2": float(model.rsquared_adj),
    }
    return out

# ------------------ Run regressions for each outcome ------------------
all_rows = []
for outcome in OUTCOMES:
    for feat in predictors:
        res = run_single_feature_ols(df, outcome, feat, CONTROLS)
        if res is not None:
            all_rows.append(res)

if not all_rows:
    raise RuntimeError("No models were fit — check your outcomes and predictors.")

results_df = pd.DataFrame(all_rows)
results_df = results_df.sort_values(["outcome", "p_value", "feature"]).reset_index(drop=True)

# Write CSVs per outcome + combined + Excel workbook
results_df.to_csv(OUT_DIR / "single_feature_regressions_ALL.csv", index=False)
for outcome in OUTCOMES:
    sub = results_df[results_df["outcome"] == outcome].copy()
    sub.to_csv(OUT_DIR / f"single_feature_regressions_{outcome}.csv", index=False)

with pd.ExcelWriter(OUT_DIR / "single_feature_regressions_by_outcome.xlsx") as xw:
    for outcome in OUTCOMES:
        sub = results_df[results_df["outcome"] == outcome]
        sub.to_excel(xw, sheet_name=outcome[:31], index=False)

print(f"[OK] wrote regression tables to: {OUT_DIR}")

# ------------------ Coefficient barplots with 95% CI ------------------
def coef_barplot(results: pd.DataFrame, outcome: str, topn: int, out_dir: Path):
    sub = results[results["outcome"] == outcome].copy()
    if sub.empty: 
        return
    # Choose topn by smallest p-value
    sub = sub.nsmallest(topn, "p_value")
    sub = sub.sort_values("coef")  # nicest for bars

    y = np.arange(len(sub))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(y, sub["coef"], xerr=1.96*sub["std_err"], alpha=0.8)
    ax.set_yticks(y, labels=sub["feature"])
    ax.set_xlabel("Coefficient (OLS, HC3 SE)")
    ax.set_title(f"Coefficients with 95% CI — {outcome}")
    for i, (coef) in enumerate(sub["coef"].round(3)):
        ax.text(coef, i, f"  b = {coef:.3f}", va="center")
    plt.tight_layout()
    fig.savefig(out_dir / f"coef_bar_{outcome}.png", dpi=200)
    plt.close(fig)

for outcome in OUTCOMES:
    coef_barplot(results_df, outcome, TOP_N_FOR_PLOT, OUT_DIR)

# ------------------ Big “feature grid” image (hist + two scatters) ------------------
def feature_grid_plot(data: pd.DataFrame, feats: list, outcomes: list, out_dir: Path, per_page: int = 12):
    """
    For each feature: col0 = histogram, col1 = scatter vs outcomes[0], col2 = scatter vs outcomes[1]
    Paginates across multiple JPEGs if needed.
    """
    feats = [f for f in feats if f in data.columns]
    pages = math.ceil(len(feats) / per_page)
    for page in range(pages):
        chunk = feats[page*per_page : (page+1)*per_page]
        nrows = len(chunk)
        ncols = 3
        fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.2*nrows), squeeze=False)
        for r, f in enumerate(chunk):
            s = pd.to_numeric(data[f], errors="coerce").dropna()
            # 0) histogram
            axes[r,0].hist(s, bins=40)
            axes[r,0].set_title(f"{f} — distribution")
            axes[r,0].set_ylabel("count")
            # 1) scatter vs first outcome
            if outcomes[0] in data.columns:
                sub = data[[f, outcomes[0]]].dropna()
                axes[r,1].scatter(sub[f], sub[outcomes[0]], s=8, alpha=0.6, color="C0")
                axes[r,1].set_title(f"{outcomes[0]} vs {f}")
                axes[r,1].set_ylabel(outcomes[0])
            # 2) scatter vs second outcome
            if len(outcomes) > 1 and outcomes[1] in data.columns:
                sub = data[[f, outcomes[1]]].dropna()
                axes[r,2].scatter(sub[f], sub[outcomes[1]], s=8, alpha=0.6, color="C3")
                axes[r,2].set_title(f"{outcomes[1]} vs {f}")
                axes[r,2].set_ylabel(outcomes[1])

            for c in range(ncols):
                axes[r,c].set_xlabel(f)

        plt.tight_layout(h_pad=1.2, w_pad=1.2)
        out_path = out_dir / f"feature_analysis_grid_p{page+1}.jpeg"
        fig.savefig(out_path, dpi=160, pil_kwargs={"quality": 90})
        plt.close(fig)

# Build the grid over all predictors
feature_grid_plot(df, predictors, OUTCOMES, OUT_DIR, per_page=12)

# ------------------ VIF for controls (sanity check) ------------------
vif_feats = [c for c in CONTROLS if c in df.columns]
if len(vif_feats) >= 2:
    X_vif = df[vif_feats].dropna().astype(float)
    X_vif = sm.add_constant(X_vif)
    vif_tbl = pd.DataFrame({
        "feature": X_vif.columns,
        "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
    })
    vif_tbl.to_csv(OUT_DIR / "vif_controls.csv", index=False)
    print("[OK] VIF written:", OUT_DIR / "vif_controls.csv")
else:
    print("[SKIP] VIF — need at least 2 control variables present.")

print("\nDone. Outputs in:", OUT_DIR)

[OK] wrote regression tables to: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level
[OK] VIF written: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level/vif_controls.csv

Done. Outputs in: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level


### fixing the graph visually (so names do not run onto each other on title)

[OK] Wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level/figures_clean/coef_bar_num_teams_clean.png
[OK] Wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level/figures_clean/coef_bar_num_funded_clean.png
[OK] Wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level/figures_clean/coef_bar_facilitator_logit_clean.png
[OK] Wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level/figures_clean/feature_analysis_grid_clean.png

All set! Figure pack at: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level/figures_clean/FigurePack.pdf
Cleaned figures in: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_meeting_level/figures_clean


In [22]:
import matplotlib.pyplot as plt
import textwrap
from matplotlib.backends.backend_pdf import PdfPages

def safe_title(text, width=25, fontsize=7):
    """Wrap long titles and shrink fontsize if needed."""
    return "\n".join(textwrap.wrap(text, width)), fontsize

# Collect features (p_* and ann_*)
features_to_plot = [c for c in df.columns 
                    if (c.startswith("ann_") or c.startswith("p_")) 
                    and pd.api.types.is_numeric_dtype(df[c])]

pdf_out = OUT_DIR / "feature_analysis_all.pdf"
with PdfPages(pdf_out) as pdf:
    # batch 9 features per page (3 rows × 3 cols)
    batch_size = 9
    for i in range(0, len(features_to_plot), batch_size):
        batch = features_to_plot[i:i+batch_size]

        fig, axes = plt.subplots(3, 3, figsize=(12, 12))
        axes = axes.flatten()

        for ax, feat in zip(axes, batch):
            if feat not in df.columns:
                continue

            # Example: histogram for distribution
            ax.hist(df[feat].dropna(), bins=30, color="steelblue", alpha=0.7)

            # safe wrapped title
            t, fs = safe_title(f"{feat} — distribution")
            ax.set_title(t, fontsize=fs)

        # Remove any unused subplots in the grid
        for ax in axes[len(batch):]:
            ax.axis("off")

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"[OK] Combined plots saved to: {pdf_out}")

[OK] Combined plots saved to: outputs_meeting_level_cleaned_titles/feature_analysis_all.pdf


# regression analysis - w/ & w/out controls

In [1]:
# =========================================
# REGRESSION COMPARISON: with vs. without controls
# File assumptions:
#   - Person-level rows
#   - Outcomes: num_teams, num_funded_teams (already created)
#   - Annotation features start with "ann_"
#   - Controls (per Evey's hunch): p_speaking_duration_sec, p_turns
#   - Optional FE: conference, year
# Outputs:
#   - ./outputs_meeting_level/compare_controls_*.csv
#   - ./outputs_meeting_level/compare_controls.xlsx
#   - ./outputs_meeting_level/compare_controls_summary.txt
#   - ./outputs_meeting_level/coef_compare_<outcome>.png
# =========================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from patsy import dmatrices

# ---------- paths ----------
DATA_CSV = Path("ALL_person_year_WITH_OUTCOMES.csv")  # use your latest person-level file
OUT_DIR  = Path("outputs_meeting_level")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- load ----------
df = pd.read_csv(DATA_CSV)

# keep only numeric ann_ features (drop all-NA or zero-variance later)
ann_feats = [c for c in df.columns if c.startswith("ann_") and pd.api.types.is_numeric_dtype(df[c])]
controls  = [c for c in ["p_speaking_duration_sec", "p_turns"] if c in df.columns]
fe_conf   = "conference" in df.columns
fe_year   = "year" in df.columns

# small hygiene
for c in ["num_teams","num_funded_teams"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# drop ann_ features that are all missing or constant
good_ann = []
for c in ann_feats:
    s = pd.to_numeric(df[c], errors="coerce")
    if s.notna().sum() > 1 and s.std(skipna=True) > 0:
        good_ann.append(c)
ann_feats = sorted(good_ann)

print(f"Annotation predictors: {len(ann_feats)}; Controls present: {controls}")

def fit_ols_hc3(data, outcome, rhs_terms):
    """Fit OLS with HC3 robust SE via patsy design; returns result and tidy df."""
    rhs = " + ".join(rhs_terms)
    formula = f"{outcome} ~ {rhs}"
    y, X = dmatrices(formula, data=data, return_type="dataframe", NA_action="drop")
    if y.shape[0] < 20:
        return None, None, formula  # too few rows to be meaningful
    model = sm.OLS(y, X).fit(cov_type="HC3")
    # tidy
    ci = model.conf_int()
    tidy = pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "p_value": model.pvalues.values,
        "ci_low": ci[0].values,
        "ci_high": ci[1].values,
    })
    tidy["N"] = int(model.nobs)
    tidy["R2"] = float(model.rsquared)
    tidy["Adj_R2"] = float(model.rsquared_adj)
    return model, tidy, formula

def make_rhs(include_controls: bool):
    rhs = ann_feats.copy()
    if include_controls:
        rhs += controls
    if fe_conf: rhs += ["C(conference)"]
    if fe_year: rhs += ["C(year)"]
    return rhs

outcomes = [c for c in ["num_teams","num_funded_teams"] if c in df.columns]

all_tables = {}
summary_lines = []

for y in outcomes:
    # (A) annotations only
    rhs_A = make_rhs(include_controls=False)
    modA, tidyA, formA = fit_ols_hc3(df, y, rhs_A)

    # (B) annotations + controls
    rhs_B = make_rhs(include_controls=True)
    modB, tidyB, formB = fit_ols_hc3(df, y, rhs_B)

    if (modA is None) or (modB is None):
        print(f"[SKIP] Not enough rows for {y}")
        continue

    # Save raw tidy tables
    a_csv = OUT_DIR / f"compare_controls_{y}_A_ann_only.csv"
    b_csv = OUT_DIR / f"compare_controls_{y}_B_ann_plus_controls.csv"
    tidyA.to_csv(a_csv, index=False)
    tidyB.to_csv(b_csv, index=False)

    # Build a merged comparison for overlapping terms (esp. ann_ terms + controls)
    dispA = tidyA[["term","coef","std_err","p_value","ci_low","ci_high","R2","Adj_R2","N"]].rename(
        columns={"coef":"coef_A","std_err":"se_A","p_value":"p_A","ci_low":"ci_low_A","ci_high":"ci_high_A",
                 "R2":"R2_A","Adj_R2":"Adj_R2_A","N":"N_A"}
    )
    dispB = tidyB[["term","coef","std_err","p_value","ci_low","ci_high","R2","Adj_R2","N"]].rename(
        columns={"coef":"coef_B","std_err":"se_B","p_value":"p_B","ci_low":"ci_low_B","ci_high":"ci_high_B",
                 "R2":"R2_B","Adj_R2":"Adj_R2_B","N":"N_B"}
    )
    merged = dispA.merge(dispB, on="term", how="outer")
    # annotate deltas
    merged["delta_coef"] = merged["coef_B"] - merged["coef_A"]
    merged["abs_pct_change_coef"] = np.where(
        merged["coef_A"].abs() > 1e-12,
        (merged["coef_B"] - merged["coef_A"]).abs() / merged["coef_A"].abs(),
        np.nan
    )
    # Add overall model deltas (repeat on every row for convenience)
    merged["delta_R2"] = merged["R2_B"].iloc[0] - merged["R2_A"].iloc[0]
    merged["delta_Adj_R2"] = merged["Adj_R2_B"].iloc[0] - merged["Adj_R2_A"].iloc[0]

    comp_csv = OUT_DIR / f"compare_controls_{y}_MERGED.csv"
    merged.to_csv(comp_csv, index=False)
    all_tables[y] = merged

    # Summary text line
    summary_lines.append(
        f"[{y}] R²: {merged['R2_A'].iloc[0]:.3f} → {merged['R2_B'].iloc[0]:.3f} "
        f"(Δ {merged['delta_R2'].iloc[0]:+.3f}); "
        f"Adj.R²: {merged['Adj_R2_A'].iloc[0]:.3f} → {merged['Adj_R2_B'].iloc[0]:.3f} "
        f"(Δ {merged['delta_Adj_R2'].iloc[0]:+.3f})."
    )

    # Quick coefficient comparison plot (controls + top 10 ann_ by |coef_B|)
    plot_terms = []
    # controls first (if present in B)
    for c in controls:
        if (merged["term"]==c).any():
            plot_terms.append(c)
    # then top 10 ann_ by absolute coef in model B
    ann_in_B = merged[merged["term"].str.startswith("ann_")].copy()
    ann_in_B["rank"] = ann_in_B["coef_B"].abs().rank(ascending=False, method="first")
    top_ann = ann_in_B.sort_values("rank").head(10)["term"].tolist()
    plot_terms += [t for t in top_ann if t not in plot_terms]

    sub = merged[merged["term"].isin(plot_terms)].copy()
    sub = sub.sort_values("coef_B")

    fig, ax = plt.subplots(figsize=(10, 6))
    y_pos = np.arange(len(sub))
    ax.errorbar(sub["coef_A"], y_pos+0.15, xerr=sub["se_A"]*1.96, fmt="o", label="Ann only (95% CI)", alpha=0.7)
    ax.errorbar(sub["coef_B"], y_pos-0.15, xerr=sub["se_B"]*1.96, fmt="s", label="Ann + controls (95% CI)", alpha=0.8)
    ax.axvline(0, color="k", lw=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(sub["term"])
    ax.set_xlabel("Coefficient (HC3 SE)")
    ax.set_title(f"Coefficient comparison — {y}")
    ax.legend(loc="lower right")
    plt.tight_layout()
    fig.savefig(OUT_DIR / f"coef_compare_{y}.png", dpi=200)
    plt.close(fig)

# one Excel with both merged tables (if available)
if all_tables:
    xlsx = OUT_DIR / "compare_controls.xlsx"
    with pd.ExcelWriter(xlsx, engine="xlsxwriter") as xw:
        for y, tab in all_tables.items():
            sheet = y[:31]
            tab.to_excel(xw, index=False, sheet_name=sheet)

# summary text
with open(OUT_DIR / "compare_controls_summary.txt","w") as f:
    f.write("Controls tested: " + ", ".join(controls) + "\n")
    f.write("\n".join(summary_lines) if summary_lines else "No models fit.\n")

print("Done. See:", OUT_DIR)

Annotation predictors: 25; Controls present: ['p_speaking_duration_sec', 'p_turns']
Done. See: outputs_meeting_level


## I should have most the information for the analysis EV was mentioning on 09/29 (refer to notes if needbe)- combining all together now

In [6]:
# =========================================
# CONSOLIDATE & SUMMARIZE (robust logit)
# =========================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import textwrap
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrices
from statsmodels.tools.sm_exceptions import PerfectSeparationError
import numpy.linalg as npl

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc
from sklearn.impute import SimpleImputer

# ---------- CONFIG ----------
DATA_CSV = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_year_WITH_OUTCOMES.csv")
OUT_DIR  = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_summary")
OUT_DIR.mkdir(parents=True, exist_ok=True)

INCLUDE_CTX  = True
RANDOM_STATE = 42
N_SPLITS     = 5

OUTCOMES = [
    ("role_facilitator", "Facilitator vs Non-facilitator"),   # binary
    ("num_teams", "Number of teams (count)"),                 # numeric
    ("num_funded_teams", "Number of funded teams (count)")    # numeric
]
CONTROLS = ["p_speaking_duration_sec", "p_turns"]

# ---------- LOAD ----------
df = pd.read_csv(DATA_CSV)
for col in ["role_facilitator","num_teams","num_funded_teams","year","sessions_total"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
if "role_facilitator" not in df.columns and "facilitator" in df.columns:
    df["role_facilitator"] = (pd.to_numeric(df["facilitator"], errors="coerce").fillna(0) > 0).astype(int)

p_feats   = [c for c in df.columns if c.startswith("p_")   and pd.api.types.is_numeric_dtype(df[c])]
ann_feats = [c for c in df.columns if c.startswith("ann_") and pd.api.types.is_numeric_dtype(df[c])]
ctx_feats = [c for c in df.columns if c.startswith("ctx_") and pd.api.types.is_numeric_dtype(df[c])]
if not INCLUDE_CTX: ctx_feats = []

def nonconstant(cols):
    out = []
    for c in cols:
        s = pd.to_numeric(df[c], errors="coerce")
        if s.notna().any() and s.std(skipna=True) > 0:
            out.append(c)
    return out
p_feats, ann_feats, ctx_feats = map(nonconstant, [p_feats, ann_feats, ctx_feats])

# ---------- helpers ----------
def wrap_label(txt, width=24):
    import textwrap
    return "\n".join(textwrap.wrap(str(txt), width=width))

def build_formula(outcome, predictors, controls=None, add_fe=True):
    rhs = []
    if controls: rhs += controls
    rhs += predictors
    if add_fe:
        if "conference" in df.columns: rhs.append("C(conference)")
        if "year" in df.columns:       rhs.append("C(year)")
    if "sessions_total" in df.columns: rhs.append("sessions_total")
    return f"{outcome} ~ " + (" + ".join(rhs) if rhs else "1")

def iqr_mask(x, k=1.5):
    x = pd.to_numeric(x, errors="coerce")
    if not x.notna().any(): return pd.Series(True, index=x.index)
    q1, q3 = np.nanpercentile(x.dropna(), [25, 75])
    lo, hi = q1 - k*(q3-q1), q3 + k*(q3-q1)
    return (x >= lo) & (x <= hi)

def trim_outliers_on_controls(df_in, controls):
    if not controls: 
        return df_in.copy(), pd.Series(False, index=df_in.index)
    m = pd.Series(True, index=df_in.index)
    for c in controls:
        if c in df_in.columns: m &= iqr_mask(df_in[c])
    return df_in.loc[m].copy(), ~m

def roc_cv_auc(data, target, X_cols):
    d = data.dropna(subset=[target]).copy()
    if d[target].nunique() < 2: return None, None, None
    y = d[target].astype(int).values
    X = d[X_cols].copy()
    numeric_transform = Pipeline(steps=[
        ("impute", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("scale", StandardScaler())
    ])
    preprocess = ColumnTransformer([("num", numeric_transform, X_cols)], remainder="drop")
    clf = Pipeline([("prep", preprocess),
                    ("clf", LogisticRegression(penalty="l2", solver="liblinear",
                                               class_weight="balanced",
                                               max_iter=2000, random_state=RANDOM_STATE))])
    cv  = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    y_prob = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")[:,1]
    fpr, tpr, _ = roc_curve(y, y_prob)
    return auc(fpr, tpr), fpr, tpr

def tidy_from_result(result, model_label):
    params = result.params; bse = result.bse; pvals = result.pvalues; conf = result.conf_int()
    out = pd.DataFrame({
        "model": model_label, "term": params.index,
        "coef": params.values, "std_err": bse.values, "p_value": pvals.values,
        "ci_low": conf[0].values, "ci_high": conf[1].values,
        "N": int(result.nobs), "llf": getattr(result, "llf", np.nan),
        "llnull": getattr(result, "llnull", np.nan)
    })
    out["pseudo_r2"] = np.where(out["llnull"].notna() & (out["llnull"]!=0),
                                1 - (out["llf"]/out["llnull"]), np.nan)
    return out

# --- NEW: full-rank safe logit with regularized fallback ---
def fit_logit_safe(formula, data):
    """
    1) Use patsy to create y, X.
    2) Drop collinear columns via QR pivoting to full rank.
    3) Fit Logit; if singular / separation, fall back to fit_regularized.
    Returns a statsmodels results object compatible with .params, .bse, etc.
    """
    # build design
    y, X = dmatrices(formula, data=data, return_type="dataframe", NA_action="drop")
    y = np.asarray(y).ravel().astype(int)
    if np.unique(y).size < 2:
        raise ValueError("Binary target has a single class after NA drop.")
    # keep intercept aside
    const = None
    if "Intercept" in X.columns:
        const = X["Intercept"].copy()
        X = X.drop(columns=["Intercept"])
    # QR select independent columns
    A = X.values
    if A.size == 0:
        X_full = pd.DataFrame({"Intercept": const}) if const is not None else pd.DataFrame({"Intercept":[1.0]*len(y)})
    else:
        rank = np.linalg.matrix_rank(A)
        if rank < A.shape[1]:
            Q, R, piv = npl.qr(A, mode="reduced", pivoting=True)
            keep_idx = piv[:rank]
            X = X.iloc[:, keep_idx]
        # reattach intercept as first col
        if const is not None:
            X.insert(0, "Intercept", const.values)
        else:
            X.insert(0, "Intercept", 1.0)
        X_full = X

    try:
        model = sm.Logit(y, X_full)
        return model.fit(disp=0)
    except (np.linalg.LinAlgError, PerfectSeparationError):
        # fallback: elastic-net style regularization
        model = sm.Logit(y, X_full)
        res = model.fit_regularized(alpha=1.0, L1_wt=0.5, disp=0)
        return res

# ---------- outliers on controls ----------
controls_present = [c for c in CONTROLS if c in df.columns]
trimmed_df, outlier_mask = trim_outliers_on_controls(df, controls_present)

# ---------- models + plots ----------
all_tables = []
for outcome, desc in OUTCOMES:
    if outcome not in df.columns: 
        continue

    is_binary = df[outcome].dropna().isin([0,1]).all()
    ann_pred = ann_feats.copy()
    if not ann_pred: 
        continue

    ctl = [c for c in CONTROLS if c in df.columns]
    f_noctl   = build_formula(outcome, ann_pred, controls=None, add_fe=True)
    f_withctl = build_formula(outcome, ann_pred, controls=ctl,  add_fe=True)

    if is_binary:
        try:
            m1 = fit_logit_safe(f_noctl,   df)
            m2 = fit_logit_safe(f_withctl, df)
            m3 = fit_logit_safe(f_noctl,   trimmed_df)
            m4 = fit_logit_safe(f_withctl, trimmed_df)
        except ValueError:
            # single-class after filtering — skip
            continue
    else:
        m1 = smf.ols(f_noctl,   data=df).fit(cov_type="HC3")
        m2 = smf.ols(f_withctl, data=df).fit(cov_type="HC3")
        m3 = smf.ols(f_noctl,   data=trimmed_df).fit(cov_type="HC3")
        m4 = smf.ols(f_withctl, data=trimmed_df).fit(cov_type="HC3")

    label = lambda s: f"{desc} — {s}"
    for res, lbl in [(m1,"ANN only, full"),(m2,"ANN+controls, full"),
                     (m3,"ANN only, trimmed"),(m4,"ANN+controls, trimmed")]:
        t = tidy_from_result(res, label(lbl))
        t.insert(0, "outcome", outcome)
        all_tables.append(t)

# save tidy
if all_tables:
    pd.concat(all_tables, ignore_index=True).to_csv(OUT_DIR / "regressions_tidy_combined.csv", index=False)

print("✓ Finished without singular-matrix errors. Outputs in:", OUT_DIR)

/opt/miniconda3/envs/gem_samp/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/miniconda3/envs/gem_samp/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/miniconda3/envs/gem_samp/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/miniconda3/envs/gem_samp/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:2443: RuntimeWarning: divide by zero encountered in log
  return np.sum(np.log(self.cdf(q * linpred)))
/opt/miniconda3/envs/gem_samp/lib/python3.11/site-packages/statsmodels/discrete/discrete_model.py:2385: RuntimeWarning: overflow encountered in exp
  return 1/(1+np.exp(-X))
/opt/miniconda3/envs/gem_samp/lib/python3.11/site-packages/statsmodels/discr

✓ Finished without singular-matrix errors. Outputs in: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/outputs_summary


## 10/01 work - continuing (fixing the plotting of the figures)

note: file path updated cause working off of MBP

In [5]:
# =========================================
# PERSON-LEVEL: single-feature OLS with controls (+ publication-ready plots)
# =========================================
from pathlib import Path
import re, math, textwrap
import numpy as np, pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# ---------- PATHS (MBP) ----------
BASE_DIR = Path("/Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini")
DATA_CSV = BASE_DIR / "facilitator-identification" / "ALL_person_year_WITH_OUTCOMES.csv"
OUT_DIR  = BASE_DIR / "facilitator-identification" / "outputs_person_level_viz"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- CONFIG ----------
CONTROLS = ["p_speaking_duration_sec", "p_turns"]             # Evey's controls
OUTCOMES = ["num_teams", "num_funded_teams"]
KEEP_PREFIXES = ("ann_", "mean_", "num_", "positive_", "meeting_length")
EXCLUDE_COLS = set(OUTCOMES) | set(CONTROLS) | {
    "person_id","participant_id","meeting_id","session_id","year","conference","sessions_total"
}
MAX_FEATURES  = 40
WRAP = 26              # wrap width (tightens label blocks)
FS_BASE = 12           # base font size
mpl.rcParams.update({"font.size": FS_BASE})

# ---------- LOAD ----------
df = pd.read_csv(DATA_CSV)

for c in set(OUTCOMES) | set(CONTROLS):
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# ---------- NAME PRETTIFIER ----------
def pretty_label(col: str, wrap=WRAP) -> str:
    """
    Make long machine names human-friendly and short.
    Rules:
      - drop leading {ann_, mean_score_, num_}
      - *_sum_score or *_sum -> "(sum)" ; *_count -> "(count)"
      - replace underscores with spaces; compact double spaces
      - lower-case body; keep acronyms; then sentence-case first word
      - wrap into multiple lines of <= wrap chars
    """
    s = col

    # outcome/controls: keep readable forms
    if s in OUTCOMES:
        return "\n".join(textwrap.wrap(s.replace("_", " "), wrap))

    # strip common prefixes once
    s = re.sub(r"^(ann_|mean_score_|num_)", "", s)

    # collapse well-known tails to compact tags
    tail_tag = None
    for pat, tag in [
        (r"_sum_score$", " (sum)"),
        (r"_sum$",       " (sum)"),
        (r"_count$",     " (count)"),
        (r"_score$",     ""),               # drop bare _score
        (r"^meeting_length$", "meeting length"),
    ]:
        if re.search(pat, s):
            s = re.sub(pat, "", s)
            if tag and tag.strip():
                tail_tag = tag
            break

    # readability
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s).strip()

    # compact common phrases
    s = (s
         .replace("coordination and decision practices", "coordination/decision practices")
         .replace("participation dynamics", "participation dynamics")
         .replace("knowledge sharing", "knowledge sharing")
         .replace("relational climate", "relational climate")
         .replace("idea management", "idea management")
         )

    if tail_tag:
        s += tail_tag

    # Title case, but keep inside parentheses lower
    def smart_title(u):
        if "(" in u:
            head, tail = re.match(r"^(.*?)(\s*\(.*\))?$", u).groups()
            head = head.title() if head else ""
            return (head + (tail or "")).strip()
        return u.title()

    s = smart_title(s)

    # wrap
    return "\n".join(textwrap.wrap(s, wrap))

# cache pretty names
PRETTY = {c: pretty_label(c) for c in df.columns}

# ---------- FEATURE PICKER ----------
cand = []
for c in df.columns:
    if c in EXCLUDE_COLS: 
        continue
    if not any(c.startswith(p) for p in KEEP_PREFIXES):
        continue
    if not pd.api.types.is_numeric_dtype(df[c]):
        continue
    if pd.to_numeric(df[c], errors="coerce").std(skipna=True) <= 0:
        continue
    if df[c].nunique(dropna=True) <= 2:
        continue
    cand.append(c)

# fallback if empty
if not cand:
    cand = [c for c in df.columns
            if c not in EXCLUDE_COLS
            and pd.api.types.is_numeric_dtype(df[c])
            and pd.to_numeric(df[c], errors="coerce").std(skipna=True) > 0
            and df[c].nunique(dropna=True) > 2]

# rank by variance
cand = sorted(cand, key=lambda x: pd.to_numeric(df[x], errors="coerce").var(skipna=True), reverse=True)[:MAX_FEATURES]
features = cand
print(f"[Info] Using {len(features)} features.")

# ---------- helpers ----------
def iqr_mask(s, k=1.5):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0: return pd.Series(True, index=s.index)
    q1, q3 = np.nanpercentile(s.dropna(), [25, 75])
    iqr = q3 - q1
    lo, hi = q1 - k*iqr, q3 + k*iqr
    return (s >= lo) & (s <= hi)

def star(p):
    return ("***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns")

def run_ols(y, x, controls, data):
    rhs = [c for c in controls if c in data.columns]
    rhs.append(x)
    fml = f"{y} ~ " + " + ".join(rhs)
    try:
        m = smf.ols(fml, data=data).fit(cov_type="HC3")
    except Exception as e:
        print(f"[WARN] {y} ~ {x} failed: {e}")
        return None
    if x not in m.params.index:
        return None
    return dict(
        outcome=y, feature=x,
        coef=float(m.params[x]),
        se=float(m.bse[x]),
        p=float(m.pvalues[x]),
        ci_low=float(m.conf_int().loc[x,0]),
        ci_high=float(m.conf_int().loc[x,1]),
        r2=float(m.rsquared),
        n=int(m.nobs)
    )

# ---------- outlier trimming on controls (diagnostic plots only) ----------
present_ctrls = [c for c in CONTROLS if c in df.columns]
mask = pd.Series(True, index=df.index)
for c in present_ctrls: mask &= iqr_mask(df[c])
df_trim = df.loc[mask].copy()
print(f"[Info] IQR trim on controls removed {len(df)-len(df_trim)} of {len(df)} rows")

# boxplots for controls (bigger labels, no overlap)
for c in present_ctrls:
    s = pd.to_numeric(df[c], errors="coerce")
    fig, ax = plt.subplots(figsize=(3.8,3.2), constrained_layout=True)
    ax.boxplot([s.dropna().values], tick_labels=[pretty_label(c, wrap=18)], showfliers=True)
    ax.set_title("Outlier check")
    fig.savefig(OUT_DIR / f"outlier_{c}.png", dpi=180)
    plt.close(fig)

# ---------- run all regressions (full data; with controls) ----------
rows = []
for y in OUTCOMES:
    if y not in df.columns: 
        print(f"[SKIP] {y} not in data"); 
        continue
    for x in features:
        res = run_ols(y, x, CONTROLS, df)
        if res: rows.append(res)

res_df = pd.DataFrame(rows)
if res_df.empty:
    print("[ERROR] No models were fit.")
else:
    res_df.to_csv(OUT_DIR / "single_feature_with_controls_FULL.csv", index=False)
    print(f"[OK] wrote {OUT_DIR / 'single_feature_with_controls_FULL.csv'}")

# ---------- convenience tables ----------
if res_df.empty: raise SystemExit

wide_r2   = res_df.pivot(index="feature", columns="outcome", values="r2")
wide_coef = res_df.pivot(index="feature", columns="outcome", values="coef")
wide_p    = res_df.pivot(index="feature", columns="outcome", values="p")

display_feats = (wide_r2.fillna(0)
                 .assign(maxR=wide_r2.max(axis=1))
                 .sort_values("maxR", ascending=False)
                 .head(16)
                 .index.tolist())

# ---------- small helpers for plotting ----------
def yticks_pretty(ax, feats):
    labels = [PRETTY.get(f, f) for f in feats]
    ax.set_yticks(range(len(feats)))
    ax.set_yticklabels(labels, ha="right")

def add_right_gutter(ax, feats, text_list, xpad=0.02):
    """Put a neat right-column with text (e.g., coef+stars or p-values)."""
    ylim = ax.get_ylim()
    xlim = ax.get_xlim()
    xr = xlim[1] + (xlim[1]-xlim[0]) * xpad
    for i, t in enumerate(text_list):
        ax.text(xr, i, t, va="center", ha="left", fontsize=FS_BASE-1)
    # expand axes to make room
    ax.set_xlim(xlim[0], xlim[1] + (xlim[1]-xlim[0]) * (xpad*10))

# ---------- FIG 1: R-squared comparison ----------
fig, ax = plt.subplots(figsize=(18,10), constrained_layout=True)
X = np.arange(len(display_feats)); w = 0.38
for i, y in enumerate(OUTCOMES):
    vals = [wide_r2.loc[f, y] if (f in wide_r2.index and y in wide_r2.columns) else np.nan
            for f in display_feats]
    bars = ax.bar(X + (i-0.5)*w, vals, width=w, label=y)
    for xi, v in zip(X + (i-0.5)*w, vals):
        if np.isfinite(v): ax.text(xi, v+(0.002 if v>=0 else -0.002), f"{v:.4f}", ha="center", va="bottom", fontsize=FS_BASE-2)
ax.set_xticks(X)
ax.set_xticklabels([PRETTY.get(f, f) for f in display_feats], rotation=35, ha="right")
ax.set_ylabel("R-squared")
ax.set_title("R-squared Comparison — Sorted by Maximum R²", loc="left", pad=8)
ax.legend()
fig.savefig(OUT_DIR / "r_squared_comparison.png", dpi=200)
plt.close(fig)

# ---------- FIG 2: R-squared heatmap ----------
fig, ax = plt.subplots(figsize=(16,10), constrained_layout=True)
sub = wide_r2.loc[display_feats, OUTCOMES]
im = ax.imshow(sub.values, cmap="viridis", aspect="auto")
ax.set_yticks(range(len(display_feats)))
ax.set_yticklabels([PRETTY.get(f, f) for f in display_feats])
ax.set_xticks(range(len(OUTCOMES)))
ax.set_xticklabels(OUTCOMES)
for i in range(sub.shape[0]):
    for j in range(sub.shape[1]):
        v = sub.iloc[i,j]
        if np.isfinite(v):
            ax.text(j, i, f"{v:.4f}", ha="center", va="center",
                    color="white" if v>0.06 else "black", fontsize=FS_BASE-2)
ax.set_title("R-squared by Feature and Outcome", loc="left", pad=8)
fig.colorbar(mpl.cm.ScalarMappable(
    norm=mpl.colors.Normalize(vmin=np.nanmin(sub.values), vmax=np.nanmax(sub.values)),
    cmap="viridis"), ax=ax, label="R-squared")
fig.savefig(OUT_DIR / "r_squared_heatmap.png", dpi=200)
plt.close(fig)

# ---------- FIG 3: side-by-side coefficient comparison (with significance color) ----------
def plot_side_by_side(outcomes, feats):
    fig, axs = plt.subplots(1, len(outcomes), figsize=(22,10), constrained_layout=True, sharex=False)
    if len(outcomes)==1: axs = [axs]
    colors = {"***":"tab:green", "**":"tab:orange", "*":"gold", "ns":"lightgray"}
    for ax, y in zip(axs, outcomes):
        sub = res_df[(res_df["outcome"]==y) & (res_df["feature"].isin(feats))].copy()
        sub["sig"] = sub["p"].apply(star)
        sub = sub.sort_values("coef")
        ax.barh(range(len(sub)), sub["coef"], color=[colors[s] for s in sub["sig"]])
        yticks_pretty(ax, sub["feature"].tolist())
        texts = [f"{c:.3f} {star(p)}" for c,p in zip(sub["coef"], sub["p"])]
        add_right_gutter(ax, sub["feature"].tolist(), texts, xpad=0.03)
        ax.axvline(0, color="k", lw=1)
        ax.set_title(f"{y}\nCoefficient Comparison with Significance")
        ax.set_xlabel("Coefficient (effect size)")
    fig.suptitle("Side-by-Side Coefficient Comparison by Outcome", fontsize=FS_BASE+2, y=0.995)
    fig.savefig(OUT_DIR / "side_by_side_coefficient_comparison.png", dpi=200)
    plt.close(fig)

plot_side_by_side(OUTCOMES, display_feats)

# ---------- FIG 4: significance matrix ----------
fig, ax = plt.subplots(figsize=(16,10), constrained_layout=True)
show = wide_p.loc[display_feats, OUTCOMES]
txt = show.applymap(star)
img = ax.imshow((show<0.05).astype(float).values, cmap="Reds", aspect="auto", vmin=0, vmax=1)
ax.set_yticks(range(len(display_feats)))
ax.set_yticklabels([PRETTY.get(f, f) for f in display_feats])
ax.set_xticks(range(len(OUTCOMES)))
ax.set_xticklabels(OUTCOMES)
for i in range(show.shape[0]):
    for j in range(show.shape[1]):
        ax.text(j, i, txt.iloc[i,j], ha="center", va="center", color="black", fontsize=FS_BASE)
ax.set_title("Significant Results Only — *** p<0.001, ** p<0.01, * p<0.05", loc="left", pad=8)
fig.savefig(OUT_DIR / "significance_matrix.png", dpi=200)
plt.close(fig)

# ---------- FIG 5: coefficient barplots with 95% CI per outcome ----------
for y in OUTCOMES:
    sub = res_df[(res_df["outcome"]==y) & (res_df["feature"].isin(display_feats))].copy()
    sub = sub.sort_values("coef")
    fig, ax = plt.subplots(figsize=(22,10), constrained_layout=True)
    ax.barh(range(len(sub)), sub["coef"], xerr=1.96*sub["se"], alpha=0.9)
    yticks_pretty(ax, sub["feature"].tolist())
    texts = [f"{c:.3f} {star(p)}" for c,p in zip(sub["coef"], sub["p"])]
    add_right_gutter(ax, sub["feature"].tolist(), texts, xpad=0.035)
    ax.axvline(0, color="k", lw=1)
    ax.set_xlabel("Coefficient (±95% CI)")
    ax.set_title(f"Coefficient Values with 95% Confidence Intervals — {y}", loc="left", pad=8)
    fig.savefig(OUT_DIR / f"coefficient_barplot_with_ci_{y}.png", dpi=200)
    plt.close(fig)

# ---------- FIG 6: p-value distribution + QQ ----------
fig, axes = plt.subplots(1,2,figsize=(18,8), constrained_layout=True)
pvals = res_df["p"].dropna().clip(0,1)
axes[0].hist(pvals, bins=20, color="skyblue", edgecolor="white")
axes[0].axvline(0.05, color="red", ls="--", label="α = 0.05")
axes[0].axvline(0.01, color="orange", ls="--", label="α = 0.01")
axes[0].set_title("Distribution of P-values"); axes[0].set_xlabel("P-value"); axes[0].set_ylabel("Frequency"); axes[0].legend()
theory = np.linspace(0,1,len(pvals), endpoint=False) + 0.5/len(pvals)
axes[1].plot(np.sort(theory), np.sort(pvals), "o")
axes[1].plot([0,1],[0,1], "r-")
axes[1].set_title("Q-Q Plot of P-values"); axes[1].set_xlabel("Theoretical quantiles"); axes[1].set_ylabel("Ordered values")
fig.suptitle("P-value Analysis", fontsize=FS_BASE+3)
fig.savefig(OUT_DIR / "p_value_analysis.png", dpi=200)
plt.close(fig)

print("Done. Plots written to:", OUT_DIR)

[Info] Using 25 features.
[Info] IQR trim on controls removed 54 of 771 rows
[OK] wrote /Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/outputs_person_level_viz/single_feature_with_controls_FULL.csv


/var/folders/wp/cn7__9416yj3c5_vcykys8hc0000gn/T/ipykernel_12432/690076572.py:290: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  txt = show.applymap(star)


Done. Plots written to: /Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/outputs_person_level_viz


## adding in the temporal component to the regression analysis

e.g. file path for e.g. is (for both MBP/Studio):

 /Users/maxchalekson/Desktop/gemini_data_analysis/data/2021MZT/session_data/2021_10_01_MZT_S9.json

 remember it's in the data folder then parse through by conference/session etc. just like before.

The output file path is 

- for MBP: '/Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal-facilitator-outputs-viz'

- for Studio: '/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal-facilitator-outputs-viz'

In [23]:
# ======================================================================
# PERSON x TEMPORAL THIRDS (first/middle/last) — controls + annotations
# Outputs:
#   - ALL_person_session_segment.csv
#   - ALL_person_year_segment.csv
#   - ALL_person_year_segment_WITH_OUTCOMES.csv  (if outcomes file found)
# Plots:
#   - seg_controls_overview.png
#   - seg_ann_counts_heatmap.png
#   - seg_ann_means_heatmap.png
#   - seg_role_facilitator_coefbars.png  (only if outcomes available)
# ======================================================================

import json, re, textwrap
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import statsmodels.formula.api as smf

# ---------- PATHS ----------
DATA_ROOT = Path("/Users/maxchalekson/Desktop/gemini_data_analysis/data")

OUT_MBP    = Path("/Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal-facilitator-outputs-viz")
OUT_STUDIO = Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal-facilitator-outputs-viz")
OUT_DIR    = OUT_MBP if OUT_MBP.exists() else OUT_STUDIO
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Outcomes (person-year) to merge if present
OUTCOME_CANDIDATES = [
    Path("/Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/ALL_person_year_WITH_OUTCOMES.csv"),
    Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/ALL_person_year_WITH_OUTCOMES.csv"),
]
OUTCOME_PATH = next((p for p in OUTCOME_CANDIDATES if p.exists()), None)

print(f"[INFO] DATA_ROOT = {DATA_ROOT}")
print(f"[INFO] OUT_DIR   = {OUT_DIR}")
print(f"[INFO] Outcomes merge: {OUTCOME_PATH if OUTCOME_PATH else '— (not found)'}")

# ---------- CONFIG ----------
WRAP = 28
SEGMENTS = ["first","middle","last"]

# Map handwritten/JSON labels → tidy families you’ve been using
ANN_ALIAS = {
    "relational_climate"                 : "ann_relational_climate",
    "participation_dynamics"             : "ann_participation_dynamics",
    "coordination_and_decision"          : "ann_coordination_and_decision_practices",
    "coordination_and_decision_practices": "ann_coordination_and_decision_practices",
    "evaluation_practices"               : "ann_evaluation_practices",
    "knowledge_sharing"                  : "ann_knowledge_sharing",
    "information_seeking"                : "ann_information_seeking",
    "integration_practices"              : "ann_integration_practices",
    "idea_management"                    : "ann_idea_management",
    "none"                               : "ann_none",
}

# ---------- HELPERS ----------
def wrap_label(s, w=WRAP): 
    return "\n".join(textwrap.wrap(str(s), width=w))

_time_pat = re.compile(r"^(?:(\d+):)?(\d{1,2}):(\d{2})(?:\.(\d+))?$")  # H:MM:SS(.ms) or M:SS(.ms)
def time_to_sec(v):
    if v is None: return None
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip()
    if s.isdigit():
        return float(int(s))
    m = _time_pat.match(s)
    if m:
        h = int(m.group(1) or 0); mm = int(m.group(2)); ss = int(m.group(3))
        frac = m.group(4)
        sec = h*3600 + mm*60 + ss
        if frac: sec += float("0."+frac)
        return float(sec)
    # "MM:SS-MM:SS"
    if "-" in s and s.count(":") >= 2:
        return None
    # "12m34s", "95s", "45m"
    s2 = s.lower()
    try:
        if s2.endswith("s") and "m" in s2:
            parts = s2.replace("s","").split("m")
            return float(int(parts[0])*60 + int(parts[1]))
        if s2.endswith("s"):
            return float(s2[:-1])
        if s2.endswith("m"):
            return float(int(s2[:-1])*60)
    except Exception:
        pass
    return None

def parse_range(s):
    if not s or "-" not in str(s): return (None, None)
    a, b = [t.strip() for t in str(s).split("-", 1)]
    return (time_to_sec(a), time_to_sec(b))

def canon_name(s):
    if not isinstance(s, str): return ""
    s = s.strip()
    if not s: return ""
    s = s.replace(", PhD","").replace(" PhD","")
    if " - " in s: s = s.split(" - ")[0]
    return s

def which_third(t, T):
    if not (isinstance(t, (int, float)) and isinstance(T, (int, float)) and T>0):
        return None
    third = T/3.0
    return "first" if t < third else ("middle" if t < 2*third else "last")

def session_duration(J):
    # Prefer explicit if present
    for k in ["total_speaking_length","meeting_duration_sec","duration_sec","total_duration_sec","meeting_length","length_sec","total_duration"]:
        v = J.get(k)
        if isinstance(v, str):
            v2 = time_to_sec(v)
            if v2 and v2>0: return v2
        if isinstance(v, (int, float)) and v>0:
            return float(v)
    # Fallback: max end over all_data
    mx = 0.0
    ad = J.get("all_data", [])
    if isinstance(ad, list):
        for ev in ad:
            if not isinstance(ev, dict): continue
            e = time_to_sec(ev.get("end_time"))
            if e is None:
                _, e = parse_range(ev.get("timestamp"))
            if e is not None:
                mx = max(mx, float(e))
    return mx if mx>0 else None

def parse_path_info(path: Path):
    conf = path.parent.parent.name
    base = path.stem
    sid = None
    for tok in base.split("_"):
        if tok.upper().startswith("S") and tok[1:].isdigit():
            sid = tok.upper(); break
    year = None
    for tok in base.split("_"):
        if len(tok)==4 and tok.isdigit():
            year = int(tok); break
    return conf, year, sid or base

# Extract speaking segments (speaker + time + duration) from all_data
def iter_speaking_segments(J):
    segs = []
    ad = J.get("all_data", [])
    if not isinstance(ad, list): 
        return segs
    for ev in ad:
        if not isinstance(ev, dict):
            continue
        who = canon_name(ev.get("speaker") or ev.get("speaker_name") or ev.get("person") or ev.get("actor"))
        if not who: 
            continue
        s = time_to_sec(ev.get("start_time"))
        e = time_to_sec(ev.get("end_time"))
        if (s is None or e is None) and isinstance(ev.get("timestamp"), str):
            s2, e2 = parse_range(ev.get("timestamp"))
            s = s if s is not None else s2
            e = e if e is not None else e2
        d = ev.get("speaking_duration")
        d = float(d) if isinstance(d, (int, float)) else None
        # resolve 2-of-3
        if d is None and s is not None and e is not None:
            d = max(0.0, float(e)-float(s))
        if s is None and e is not None and d is not None:
            s = float(e)-float(d)
        if e is None and s is not None and d is not None:
            e = float(s)+float(d)
        if d is None:
            continue
        # midpoint
        if s is not None and e is not None:
            mid = 0.5*(float(s)+float(e))
        elif s is not None:
            mid = float(s)+float(d)/2.0
        elif e is not None:
            mid = float(e)-float(d)/2.0
        else:
            mid = float(d)/2.0
        segs.append((who, float(d), float(mid)))
    return segs

# Extract annotation items attached to all_data rows: {"annotations": {...}}
def iter_annotations(J):
    """
    Yields (who, family, score, mid_sec)
    Accepts "annotations": { "Relational Climate": {"score":2, ...}, ... }
    Family keys are normalized via ANN_ALIAS.
    """
    out = []
    ad = J.get("all_data", [])
    if not isinstance(ad, list): 
        return out
    for ev in ad:
        if not isinstance(ev, dict):
            continue
        anns = ev.get("annotations")
        if not isinstance(anns, dict) or not anns:
            continue
        who = canon_name(ev.get("speaker") or ev.get("speaker_name") or ev.get("person") or ev.get("actor"))
        # time midpoints (same logic as speaking segs)
        s = time_to_sec(ev.get("start_time"))
        e = time_to_sec(ev.get("end_time"))
        if (s is None or e is None) and isinstance(ev.get("timestamp"), str):
            s2, e2 = parse_range(ev.get("timestamp"))
            s = s if s is not None else s2
            e = e if e is not None else e2
        d = ev.get("speaking_duration")
        d = float(d) if isinstance(d, (int, float)) else None
        if s is not None and e is not None:
            mid = 0.5*(float(s)+float(e))
        elif s is not None and d is not None:
            mid = float(s)+float(d)/2.0
        elif e is not None and d is not None:
            mid = float(e)-float(d)/2.0
        else:
            mid = None

        # iterate categories
        for raw_cat, payload in anns.items():
            cat = str(raw_cat).strip().lower().replace(" ", "_")
            fam = ANN_ALIAS.get(cat)
            if fam is None:
                continue
            score = None
            if isinstance(payload, dict):
                sc = payload.get("score")
                if isinstance(sc, (int, float)):
                    score = float(sc)
            out.append((who, fam, score, mid))
    return out

# ---------- SCAN ----------
rows = []
files_seen = 0
skipped_no_duration = 0
skipped_no_segments = 0

conf_dirs = [p for p in DATA_ROOT.glob("*") if (p.is_dir() and (p/"session_data").exists())]
print("[INFO] conferences:", [d.name for d in conf_dirs])

for conf_dir in conf_dirs:
    for js in (conf_dir/"session_data").glob("*.json"):
        files_seen += 1
        try:
            J = json.loads(js.read_text())
        except Exception as e:
            print(f"[WARN] read error {js.name}: {e}")
            continue

        T = session_duration(J)
        if not T or T <= 0:
            skipped_no_duration += 1
            if skipped_no_duration <= 5:
                print(f"[SKIP:no-duration] {js.name} keys={list(J.keys())[:6]}")
            continue

        conference, year, session_id = parse_path_info(js)

        # Controls (speaking) per person×third
        speak_segs = iter_speaking_segments(J)
        if not speak_segs:
            skipped_no_segments += 1
            if skipped_no_segments <= 8:
                print(f"[SKIP:no-speaking-segments] {js.name}")
            # we still might have annotations with timestamps—so don't continue yet

        controls = defaultdict(lambda: {"p_speaking_duration_sec":0.0,"p_turns":0})
        for who, dur, mid in speak_segs:
            seg = which_third(mid, T) if mid is not None else "middle"
            if not seg: seg = "middle"
            controls[(who, seg)]["p_speaking_duration_sec"] += float(dur)
            controls[(who, seg)]["p_turns"] += 1

        # Annotation families per person×third
        anns = iter_annotations(J)
        ann_aggs = defaultdict(lambda: defaultdict(list))  # {(who,seg)[fam]} -> [scores]
        for who, fam, score, mid in anns:
            if not who or fam is None:
                continue
            seg = which_third(mid, T) if mid is not None else "middle"
            if not seg: seg = "middle"
            ann_aggs[(who, seg)][fam].append(score)

        # union of people×segment appearing in either controls or anns
        keys = set(controls.keys()) | set(k for k in ann_aggs.keys())
        if not keys:
            continue

        for (who, seg) in keys:
            row = {
                "person_name": who,
                "conference": conference,
                "year": year,
                "session_id": session_id,
                "segment": seg,
                "session_duration_sec": float(T),
                "p_speaking_duration_sec": float(controls[(who,seg)]["p_speaking_duration_sec"]),
                "p_turns": int(controls[(who,seg)]["p_turns"]),
            }
            # explode annotation families
            fams = ann_aggs.get((who,seg), {})
            for fam, scores in fams.items():
                clean = [s for s in scores if s is not None]
                row[f"{fam}_count"] = int(len(scores))
                row[f"{fam}_sum_score"] = float(np.nansum(clean)) if clean else 0.0
                row[f"{fam}_mean_score"] = (float(np.nanmean(clean)) if clean else np.nan)
            rows.append(row)

print(f"[INFO] scanned={files_seen} | skipped no-duration={skipped_no_duration} | skipped no-speaking-seg={skipped_no_segments}")

# ---------- WRITE SESSION-SEGMENT ----------
pss = pd.DataFrame(rows)
if pss.empty:
    raise RuntimeError("No person×session×third rows built. Check that your JSON has all_data with speaker + time and/or annotations with time.")

pss["year"] = pd.to_numeric(pss["year"], errors="coerce").astype("Int64")

# fill numeric family cols
for c in pss.columns:
    if c.endswith("_count"):
        pss[c] = pd.to_numeric(pss[c], errors="coerce").fillna(0).astype(int)
    elif c.endswith("_sum_score"):
        pss[c] = pd.to_numeric(pss[c], errors="coerce").fillna(0.0).astype(float)
    elif c.endswith("_mean_score"):
        pss[c] = pd.to_numeric(pss[c], errors="coerce")

pss_path = OUT_DIR / "ALL_person_session_segment.csv"
pss.to_csv(pss_path, index=False)
print(f"[OK] wrote {pss_path}  rows={len(pss)}")

# ---------- WRITE YEAR-SEGMENT (aggregate sessions) ----------
sum_cols  = [c for c in pss.columns if c.endswith("_count") or c.endswith("_sum_score") or c in ["p_speaking_duration_sec","p_turns"]]
mean_cols = [c for c in pss.columns if c.endswith("_mean_score")]

def agg_mean_ignore_na(x):
    x = pd.to_numeric(x, errors="coerce")
    return float(np.nanmean(x)) if np.isfinite(x).any() else np.nan

agg_dict = {**{c:"sum" for c in sum_cols}, **{c:agg_mean_ignore_na for c in mean_cols}}

pys = (pss.groupby(["person_name","conference","year","segment"], dropna=False, as_index=False)
          .agg(agg_dict))

pys_path = OUT_DIR / "ALL_person_year_segment.csv"
pys.to_csv(pys_path, index=False)
print(f"[OK] wrote {pys_path}  rows={len(pys)}")

# ---------- OPTIONAL MERGE OUTCOMES ----------
pys_out = pys.copy()
if OUTCOME_PATH and OUTCOME_PATH.exists():
    base = pd.read_csv(OUTCOME_PATH)
    keep = [c for c in ["role_facilitator","num_teams","num_funded_teams","sessions_total"] if c in base.columns]
    if {"person_name","year"}.issubset(base.columns) and keep:
        merged = base[["person_name","year"]+keep].drop_duplicates(["person_name","year"])
        pys_out = pys_out.merge(merged, on=["person_name","year"], how="left")
        print(f"[OK] merged outcomes: {keep}")
    else:
        print("[WARN] outcomes file found but keys/cols missing; skipping merge.")
else:
    print("[INFO] no outcomes file; skipping merge.")

pys_out_path = OUT_DIR / "ALL_person_year_segment_WITH_OUTCOMES.csv"
pys_out.to_csv(pys_out_path, index=False)
print(f"[OK] wrote {pys_out_path}  rows={len(pys_out)}")

# ---------- QUICK VISUALS ----------
def maybe_cols(df, cols):
    return [c for c in cols if c in df.columns]

# 1) Controls overview by segment
fig, ax = plt.subplots(1,2, figsize=(11,4))
ctrl = pys_out.groupby("segment")[["p_speaking_duration_sec","p_turns"]].sum().reindex(SEGMENTS)
ctrl["p_speaking_duration_hours"] = ctrl["p_speaking_duration_sec"]/3600.0
ax[0].bar(ctrl.index, ctrl["p_speaking_duration_hours"])
ax[0].set_title("Total speaking duration by segment"); ax[0].set_ylabel("Hours")
ax[1].bar(ctrl.index, ctrl["p_turns"])
ax[1].set_title("Total speaking turns by segment"); ax[1].set_ylabel("Turns")
fig.tight_layout()
fig.savefig(OUT_DIR / "seg_controls_overview.png", dpi=170)
plt.close(fig)

# 2) Heatmaps of annotation counts/means by segment (top N families)
families = sorted({c.rsplit("_",2)[0] for c in pys_out.columns if c.endswith("_count")})
topN = min(12, len(families))
fam_for_plot = families[:topN]

def heat(data, value_col, fname, title):
    if not fam_for_plot:
        return
    mat = []
    for fam in fam_for_plot:
        col = f"{fam}_{value_col}"
        if col not in data.columns:
            mat.append([np.nan]*len(SEGMENTS))
        else:
            s = data.groupby("segment")[col].mean().reindex(SEGMENTS)
            mat.append(list(s.values))
    M = np.array(mat, dtype=float)
    fig, ax = plt.subplots(figsize=(12, 0.45*len(fam_for_plot)+2))
    im = ax.imshow(M, aspect="auto", cmap="viridis")
    ax.set_yticks(range(len(fam_for_plot)))
    ax.set_yticklabels([wrap_label(f, 36) for f in fam_for_plot])
    ax.set_xticks(range(len(SEGMENTS))); ax.set_xticklabels(SEGMENTS)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if np.isfinite(M[i,j]):
                ax.text(j, i, f"{M[i,j]:.2f}", ha="center", va="center", color=("white" if M[i,j]>np.nanmean(M) else "black"), fontsize=9)
    ax.set_title(title)
    fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=np.nanmin(M), vmax=np.nanmax(M)), cmap="viridis"),
                 ax=ax, label=value_col)
    fig.tight_layout()
    fig.savefig(OUT_DIR / fname, dpi=170)
    plt.close(fig)

heat(pys_out, "count", "seg_ann_counts_heatmap.png", "Annotation counts — mean per person by segment")
heat(pys_out, "mean_score", "seg_ann_means_heatmap.png", "Annotation mean scores — mean per person by segment")

# 3) If role_facilitator exists: simple logit with segment interactions per family
if "role_facilitator" in pys_out.columns:
    rows_l = []
    for fam in families:
        fam_col = f"{fam}_count"
        if fam_col not in pys_out.columns:
            continue
        dfm = pys_out.dropna(subset=["role_facilitator"]).copy()
        dfm["role_facilitator"] = pd.to_numeric(dfm["role_facilitator"], errors="coerce").fillna(0).astype(int)
        formula = f"role_facilitator ~ {fam_col} * C(segment)"
        if "conference" in dfm.columns:
            formula += " + C(conference)"
        if "year" in dfm.columns:
            formula += " + C(year)"
        try:
            m = smf.logit(formula, data=dfm).fit(disp=0)
        except Exception:
            continue
        # gather the family term & its segment interactions
        coefs = m.params
        ses   = m.bse
        take = [idx for idx in coefs.index if idx.startswith(fam_col)]
        for idx in take:
            rows_l.append({
                "family": fam,
                "term": idx.replace(fam_col, fam),
                "coef": float(coefs[idx]),
                "se": float(ses[idx]),
                "p": float(m.pvalues[idx])
            })
    if rows_l:
        coefdf = pd.DataFrame(rows_l)
        # keep the most informative terms per family (main + interactions)
        coefdf["abscoef"] = coefdf["coef"].abs()
        keep = coefdf.sort_values(["family","abscoef"], ascending=[True, False]).groupby("family").head(3)
        keep = keep.sort_values("coef")
        fig, ax = plt.subplots(figsize=(12, max(3, 0.5*keep.shape[0])))
        ylabels = [wrap_label(f"{r['family']} • {r['term']}", 36) for _, r in keep.iterrows()]
        ax.barh(ylabels, keep["coef"], xerr=1.96*keep["se"], alpha=0.9)
        ax.axvline(0, color="k", lw=1)
        for yy, (c, p) in enumerate(zip(keep["coef"], keep["p"])):
            ax.text(c + (0.02 if c>=0 else -0.02), yy, f"{c:.3f} ({'***' if p<1e-3 else '**' if p<1e-2 else '*' if p<0.05 else 'ns'})",
                    va="center", ha="left" if c>=0 else "right", fontsize=9)
        ax.set_title("Facilitation logit — family counts × segment (coef ± 95% CI)")
        ax.set_xlabel("Log-odds")
        fig.tight_layout()
        fig.savefig(OUT_DIR / "seg_role_facilitator_coefbars.png", dpi=170)
        plt.close(fig)

print("\n[DONE] Temporal thirds (person-level) complete.")
print("CSV outputs:")
print("  -", OUT_DIR / "ALL_person_session_segment.csv")
print("  -", OUT_DIR / "ALL_person_year_segment.csv")
print("  -", OUT_DIR / "ALL_person_year_segment_WITH_OUTCOMES.csv")
print("Plots:")
print("  -", OUT_DIR / "seg_controls_overview.png")
print("  -", OUT_DIR / "seg_ann_counts_heatmap.png")
print("  -", OUT_DIR / "seg_ann_means_heatmap.png")
print("  -", OUT_DIR / "seg_role_facilitator_coefbars.png", "(if outcomes present)")

[INFO] DATA_ROOT = /Users/maxchalekson/Desktop/gemini_data_analysis/data
[INFO] OUT_DIR   = /Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal-facilitator-outputs-viz
[INFO] Outcomes merge: — (not found)
[INFO] conferences: ['2021ABI', '2021SLU', '2022MND', '2020NES', '2021CMC', '2021MND', '2021MZT', '2021NES']
[INFO] scanned=157 | skipped no-duration=0 | skipped no-speaking-seg=0
[OK] wrote /Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal-facilitator-outputs-viz/ALL_person_session_segment.csv  rows=4450
[OK] wrote /Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal-facilitator-outputs-viz/ALL_person_year_segment.csv  rows=1788
[INFO] no outcomes file; skipp

/var/folders/wp/cn7__9416yj3c5_vcykys8hc0000gn/T/ipykernel_12432/4119009273.py:417: RuntimeWarning: All-NaN slice encountered
  fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=np.nanmin(M), vmax=np.nanmax(M)), cmap="viridis"),
/var/folders/wp/cn7__9416yj3c5_vcykys8hc0000gn/T/ipykernel_12432/4119009273.py:417: RuntimeWarning: All-NaN slice encountered
  fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=np.nanmin(M), vmax=np.nanmax(M)), cmap="viridis"),



[DONE] Temporal thirds (person-level) complete.
CSV outputs:
  - /Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal-facilitator-outputs-viz/ALL_person_session_segment.csv
  - /Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal-facilitator-outputs-viz/ALL_person_year_segment.csv
  - /Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal-facilitator-outputs-viz/ALL_person_year_segment_WITH_OUTCOMES.csv
Plots:
  - /Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal-facilitator-outputs-viz/seg_controls_overview.png
  - /Users/maxchalekson/Northwestern Univers

### confirming csv layouts (sanity check)

In [24]:
import pandas as pd
from pathlib import Path

base = Path("/Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal-facilitator-outputs-viz")

pss  = pd.read_csv(base / "ALL_person_session_segment.csv")
pys  = pd.read_csv(base / "ALL_person_year_segment.csv")
pysw = pd.read_csv(base / "ALL_person_year_segment_WITH_OUTCOMES.csv")

def peek(df, name):
    ann_cols = sorted([c for c in df.columns if c.startswith("ann_")])
    print(f"\n=== {name} ===")
    print("rows:", len(df))
    print("has segment?:", "segment" in df.columns, "unique:", df["segment"].unique()[:6] if "segment" in df.columns else None)
    print("has outcomes?:", [c for c in ["role_facilitator","num_teams","num_funded_teams","sessions_total"] if c in df.columns])
    print("sample ann cols:", ann_cols[:8])
    print("n ann cols:", len(ann_cols))

peek(pss,  "session-level (person×session×segment)")
peek(pys,  "year-level (person×year×segment)")
peek(pysw, "year-level WITH outcomes")


=== session-level (person×session×segment) ===
rows: 4450
has segment?: True unique: ['middle' 'last' 'first']
has outcomes?: []
sample ann cols: ['ann_coordination_and_decision_practices_count', 'ann_coordination_and_decision_practices_mean_score', 'ann_coordination_and_decision_practices_sum_score', 'ann_evaluation_practices_count', 'ann_evaluation_practices_mean_score', 'ann_evaluation_practices_sum_score', 'ann_idea_management_count', 'ann_idea_management_mean_score']
n ann cols: 27

=== year-level (person×year×segment) ===
rows: 1788
has segment?: True unique: ['first' 'last' 'middle']
has outcomes?: []
sample ann cols: ['ann_coordination_and_decision_practices_count', 'ann_coordination_and_decision_practices_mean_score', 'ann_coordination_and_decision_practices_sum_score', 'ann_evaluation_practices_count', 'ann_evaluation_practices_mean_score', 'ann_evaluation_practices_sum_score', 'ann_idea_management_count', 'ann_idea_management_mean_score']
n ann cols: 27

=== year-level WITH

## temporal analysis -- running the graphs

In [8]:
# ============================================================
# TEMPORAL (person-year-segment) — single-feature OLS w/ controls
# - Finds temporal CSVs in the *parent* folder you specified or the old viz folder
# - Robustly merges outcomes
# - Produces results CSV + plots (R² heatmaps, coef±CI bars, sig matrices)
# ============================================================

from pathlib import Path
import numpy as np, pandas as pd, textwrap
import matplotlib.pyplot as plt
import matplotlib as mpl
import statsmodels.formula.api as smf

# ---------------- PATHS ----------------
ROOTS = [
    Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini"),
    Path("/Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini"),
]

# New parent folder (where you moved the CSVs)
TEMP_PARENTS = [
    Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant"),
    Path("/Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant"),
]

# Old viz folder (still searched just in case)
TEMP_VIZES = [
    Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal-facilitator-outputs-viz"),
    Path("/Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal-facilitator-outputs-viz"),
]

OUTCOME_REL = "facilitator-identification/facilitator-identification-significant/ALL_person_year_WITH_OUTCOMES.csv"

def find_first(*candidates):
    for p in candidates:
        if p and p.exists():
            return p
    return None

def find_file(name, roots):
    for r in roots:
        if r and r.exists():
            hits = list(r.rglob(name))
            if hits:
                # Prefer a hit inside "temporal-facilitator-significant" if multiple
                hits.sort(key=lambda x: 0 if "temporal-facilitator-significant" in str(x) else 1)
                return hits[0]
    return None

# Look in the new parent first, then old viz folder
SEARCH_ROOTS = [*TEMP_PARENTS, *TEMP_VIZES]

PATH_WITH = find_file("ALL_person_year_segment_WITH_OUTCOMES.csv", SEARCH_ROOTS)
PATH_YEAR = find_file("ALL_person_year_segment.csv",              SEARCH_ROOTS)
PATH_SESS = find_file("ALL_person_session_segment.csv",           SEARCH_ROOTS)

OUTCOME_PATH = None
for base in ROOTS:
    cand = base / OUTCOME_REL
    if cand.exists():
        OUTCOME_PATH = cand
        break

OUT_DIR = find_first(*TEMP_PARENTS) or find_first(*TEMP_VIZES) or ROOTS[0]
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("[INFO] OUT_DIR:", OUT_DIR)
print("[INFO] located:")
print("  - YEAR_SEGMENT_WITH_OUTCOMES:", PATH_WITH)
print("  - YEAR_SEGMENT:", PATH_YEAR)
print("  - SESSION_SEG :", PATH_SESS)
print("  - OUTCOMES    :", OUTCOME_PATH)

# ---------------- LOAD / MERGE ----------------
def load_temporal_base():
    if PATH_WITH is not None:
        return pd.read_csv(PATH_WITH)
    if PATH_YEAR is not None:
        return pd.read_csv(PATH_YEAR)
    if PATH_SESS is not None:
        pss = pd.read_csv(PATH_SESS)
        # aggregate to person-year-segment (keep speaking controls)
        return (pss.groupby(["person_name","conference","year","segment"], dropna=False, as_index=False)
                   .sum(numeric_only=True))
    raise FileNotFoundError("Could not find any temporal CSVs (WITH, YEAR, or SESSION).")

merged = load_temporal_base()

# If outcomes missing/empty, try to merge them
needs_merge = not set(["num_teams","num_funded_teams"]).intersection(merged.columns) or \
              merged[["num_teams","num_funded_teams"]].isna().all().all() if \
              set(["num_teams","num_funded_teams"]).issubset(merged.columns) else True

if needs_merge and OUTCOME_PATH is not None:
    out = pd.read_csv(OUTCOME_PATH)
    if {"person_name","year"}.issubset(out.columns):
        keep_out = [c for c in ["person_name","year","num_teams","num_funded_teams",
                                "role_facilitator","sessions_total"] if c in out.columns]
        out_small = out[keep_out].drop_duplicates(subset=["person_name","year"])
        merged = merged.merge(out_small, on=["person_name","year"], how="left")
        print("[INFO] merged outcomes into temporal table.")
    else:
        print("[WARN] outcomes file missing person_name/year; proceeding without merge.")

# hygiene
for c in ["num_teams","num_funded_teams","p_turns","p_speaking_duration_sec","sessions_total","year"]:
    if c in merged.columns:
        merged[c] = pd.to_numeric(merged[c], errors="coerce")

if "segment" in merged.columns:
    merged["segment"] = merged["segment"].astype(str).str.lower().str.strip()
    merged = merged[merged["segment"].isin(["first","middle","last"])]

if "person_name" in merged.columns:
    merged["person_name"] = merged["person_name"].astype(str).str.strip()

print("[INFO] temporal rows:", len(merged))

# ---------------- CONFIG ----------------
CONTROLS = [c for c in ["p_speaking_duration_sec","p_turns"] if c in merged.columns]
OUTCOMES = [c for c in ["num_teams","num_funded_teams"] if c in merged.columns]
if not OUTCOMES:
    cols = list(merged.columns)
    raise RuntimeError(f"No numeric outcomes after merge. Columns present sample: {cols[:50]}")

SEGMENTS = ["first","middle","last"]
WRAP = 26
MIN_NON_NULL = 10

# ---------------- FEATURES ----------------
ann_counts = sorted([c for c in merged.columns if c.startswith("ann_") and c.endswith("_count")])
ann_means  = sorted([c for c in merged.columns if c.startswith("ann_") and c.endswith("_mean_score")])

def fam_of(col):
    if col.endswith("_count"): return col[:-len("_count")]
    if col.endswith("_mean_score"): return col[:-len("_mean_score")]
    return col

families = sorted(set(map(fam_of, ann_counts + ann_means)))

def has_variance(s, min_n=MIN_NON_NULL):
    s = pd.to_numeric(s, errors="coerce")
    return (s.notna().sum() >= min_n) and (s.std(skipna=True) > 0)

def choose_feature_for_family(df_seg, fam, min_n=MIN_NON_NULL):
    cnt  = f"{fam}_count"
    mean = f"{fam}_mean_score"
    # Prefer counts; if too sparse, fall back to mean_score
    if cnt  in df_seg.columns and has_variance(df_seg[cnt],  min_n): return cnt
    if mean in df_seg.columns and has_variance(df_seg[mean], min_n): return mean
    return None

# ---------------- MODEL ----------------
def run_ols(y, x, data):
    rhs = [x] + [c for c in CONTROLS if c in data.columns]
    if "conference" in data.columns: rhs.append("C(conference)")
    if "year" in data.columns:       rhs.append("C(year)")
    formula = f"{y} ~ " + " + ".join(rhs)
    try:
        m = smf.ols(formula, data=data).fit(cov_type="HC3")
    except Exception:
        return None
    if x not in m.params.index:
        return None
    ci = m.conf_int().loc[x]
    return dict(
        outcome=y, feature=x,
        coef=float(m.params[x]), se=float(m.bse[x]), p=float(m.pvalues[x]),
        ci_low=float(ci.iloc[0]), ci_high=float(ci.iloc[1]),
        r2=float(m.rsquared), n=int(m.nobs)
    )

# ---------------- RUN ----------------
all_rows = []
for seg in SEGMENTS:
    dseg = merged[merged["segment"] == seg].copy()
    if dseg.empty:
        print(f"[SKIP] {seg}: no rows."); continue

    chosen = []
    for fam in families:
        xcol = choose_feature_for_family(dseg, fam, min_n=MIN_NON_NULL)
        if xcol:
            chosen.append(xcol)
    print(f"[INFO] {seg}: eligible predictors = {len(chosen)} / {len(families)}")

    for y in OUTCOMES:
        dy = dseg.dropna(subset=[y]).copy()
        if dy[y].nunique(dropna=True) <= 1:
            print(f"[SKIP] {seg} — {y}: no variance in outcome."); continue
        elig = [x for x in chosen if has_variance(dy[x], min_n=MIN_NON_NULL)]
        print(f"   {y}: usable predictors = {len(elig)}")
        for x in elig:
            res = run_ols(y, x, dy)
            if res:
                res["segment"] = seg
                all_rows.append(res)

res = pd.DataFrame(all_rows)
if res.empty:
    raise RuntimeError("No models fit. Lower MIN_NON_NULL to 5, or inspect sparsity/outcome variance.")

out_csv = OUT_DIR / "temporal_single_feature_with_controls.csv"
res.to_csv(out_csv, index=False)
print(f"[OK] wrote {out_csv} (rows={len(res)})")

# ---------------- PLOTS ----------------
def wrap_lab(s, width=WRAP): 
    return "\n".join(textwrap.wrap(str(s), width=width))

def top_features_for_segment(res_df, seg, top_k=18):
    sub = res_df[res_df["segment"]==seg]
    if sub.empty: return []
    r2w = sub.pivot(index="feature", columns="outcome", values="r2").fillna(0.0)
    r2w["maxR"] = r2w.max(axis=1)
    return r2w.sort_values("maxR", ascending=False).head(top_k).index.tolist()

# 1) R² heatmaps per segment
for seg in SEGMENTS:
    sub = res[res["segment"]==seg]
    if sub.empty: continue
    r2w = sub.pivot(index="feature", columns="outcome", values="r2").fillna(0.0)
    feats = top_features_for_segment(res, seg, top_k=18)
    r2w = r2w.loc[[f for f in feats if f in r2w.index]]
    if r2w.empty: continue
    fig, ax = plt.subplots(figsize=(12, max(6, 0.4*len(r2w))))
    im = ax.imshow(r2w.values, cmap="viridis", aspect="auto")
    ax.set_yticks(range(len(r2w.index)))
    ax.set_yticklabels([wrap_lab(f, 28) for f in r2w.index])
    ax.set_xticks(range(len(r2w.columns)))
    ax.set_xticklabels(r2w.columns)
    for i in range(r2w.shape[0]):
        for j in range(r2w.shape[1]):
            val = r2w.values[i,j]
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", 
                    color="white" if val>0.04 else "black", fontsize=8)
    ax.set_title(f"R² by Feature × Outcome — {seg.title()}")
    fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=np.nanmin(r2w.values),
                                                                 vmax=np.nanmax(r2w.values)),
                                       cmap="viridis"),
                 ax=ax, label="R²")
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"r2_heatmap_{seg}.png", dpi=170)
    plt.close(fig)

# 2) Coefficient barplots with 95% CI per outcome × segment
for seg in SEGMENTS:
    for y in OUTCOMES:
        sub = res[(res["segment"]==seg) & (res["outcome"]==y)].copy()
        if sub.empty: continue
        feats = top_features_for_segment(res, seg, top_k=18)
        sub = sub[sub["feature"].isin(feats)].sort_values("coef")
        fig, ax = plt.subplots(figsize=(15, max(6, 0.35*len(sub))))
        ylabels = [wrap_lab(f, 30) for f in sub["feature"]]
        ax.barh(ylabels, sub["coef"], xerr=1.96*sub["se"], alpha=0.9)
        ax.axvline(0, color="k", lw=1)
        ax.set_xlabel("Coefficient (±95% CI)")
        ax.set_title(f"Coef ±95% CI — {y} — {seg.title()}")
        fig.tight_layout()
        fig.savefig(OUT_DIR / f"coef_bar_{y}_{seg}.png", dpi=170)
        plt.close(fig)

# 3) Significance matrix (***, **, *, ns) per segment
def p_to_star(p):
    return ("***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns")

for seg in SEGMENTS:
    sub = res[res["segment"]==seg].copy()
    if sub.empty: continue
    feats = top_features_for_segment(res, seg, top_k=18)
    show = sub[sub["feature"].isin(feats)]
    if show.empty: continue
    pw = show.pivot(index="feature", columns="outcome", values="p")
    txt = pw.applymap(p_to_star)
    fig, ax = plt.subplots(figsize=(10, max(5, 0.35*len(txt))))
    img = ax.imshow((pw < 0.05).astype(float).values, cmap="Reds", aspect="auto", vmin=0, vmax=1)
    ax.set_yticks(range(len(txt.index))); ax.set_yticklabels([wrap_lab(f, 28) for f in txt.index])
    ax.set_xticks(range(len(txt.columns))); ax.set_xticklabels(txt.columns)
    for i in range(txt.shape[0]):
        for j in range(txt.shape[1]):
            ax.text(j, i, txt.values[i,j], ha="center", va="center", color="black", fontsize=9)
    ax.set_title(f"Significance (α=0.05) — {seg.title()}")
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"significance_matrix_{seg}.png", dpi=170)
    plt.close(fig)

print("[DONE] Wrote:")
print(f" - {out_csv.name}")
print(" - r2_heatmap_first/middle/last.png")
print(" - coef_bar_<outcome>_<segment>.png")
print(" - significance_matrix_<segment>.png")

[INFO] OUT_DIR: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant
[INFO] located:
  - YEAR_SEGMENT_WITH_OUTCOMES: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/ALL_person_year_segment_WITH_OUTCOMES.csv
  - YEAR_SEGMENT: /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/ALL_person_year_segment.csv
  - SESSION_SEG : /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/ALL_person_session_segment.csv
  - OUTCOMES    : /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/facilitator-identification-significant/ALL_person_year_WITH_OUTCOMES.csv
[INFO] merged outcomes into temporal table.
[INFO] temporal rows: 1788
[INFO] first: eligible predictors = 9 / 9
   num_teams: usable pr

/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_18475/1353865037.py:274: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  txt = pw.applymap(p_to_star)
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_18475/1353865037.py:274: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  txt = pw.applymap(p_to_star)
/var/folders/17/lphw45ds14n5t74zwjdfybx40000gn/T/ipykernel_18475/1353865037.py:274: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  txt = pw.applymap(p_to_star)


[DONE] Wrote:
 - temporal_single_feature_with_controls.csv
 - r2_heatmap_first/middle/last.png
 - coef_bar_<outcome>_<segment>.png
 - significance_matrix_<segment>.png


## quick confirming differences b/w roles (temporally)

- facilitators or not

- in-teams or funded teams?

    - (remember there's three ways to parse through this)

In [1]:
# ============================================================
# TEMPORAL ROLE/TEAM DIFFERENCES (answers Evey's question)
# - Compare facilitators vs non-facilitators within each segment
# - Compare people with any funded team vs none within each segment
# - Uses annotation *_count features (same families as before)
# - Controls: speaking duration, turns, conf/year FE in regressions
# - Outputs CSVs + barplots highlighting biggest effects
# ============================================================

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# ---------------- PATHS ----------------
ROOTS = [
    Path("/Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini"),
    Path("/Users/maxchalekson/Northwestern University/Summer-2025/NICO/NICO Research/NICO_human-gemini"),
]

TEMP_REL = "facilitator-identification/temporal-facilitator-significant"
OUTCOME_REL = "facilitator-identification/facilitator-identification-significant/ALL_person_year_WITH_OUTCOMES.csv"

BASE = next((b for b in ROOTS if b.exists()), ROOTS[0])
TEMP_DIR = BASE / TEMP_REL
OUT_DIR  = TEMP_DIR  # write outputs here

# Prefer WITH_OUTCOMES; fall back to YEAR_SEGMENT + merge
CANDS = [
    TEMP_DIR / "ALL_person_year_segment_WITH_OUTCOMES.csv",
    TEMP_DIR / "ALL_person_year_segment.csv",
]
TEMP_PATH = next((p for p in CANDS if p.exists()), None)
OUTCOME_PATH = next((BASE / OUTCOME_REL for _ in [0] if (BASE / OUTCOME_REL).exists()), None)

if TEMP_PATH is None:
    raise FileNotFoundError("Could not find temporal CSVs in\n" + "\n".join(map(str,CANDS)))

df = pd.read_csv(TEMP_PATH)

# If outcomes not present, merge them
needed_outs = {"num_teams","num_funded_teams"}
if not needed_outs.intersection(df.columns) and OUTCOME_PATH is not None:
    outs = pd.read_csv(OUTCOME_PATH)
    keep = [c for c in ["person_name","year","role_facilitator","num_teams","num_funded_teams","sessions_total"] if c in outs.columns]
    outs_small = outs[keep].drop_duplicates(subset=["person_name","year"])
    df = df.merge(outs_small, on=["person_name","year"], how="left")

# ---------------- HYGIENE ----------------
for c in ["num_teams","num_funded_teams","p_speaking_duration_sec","p_turns","year","role_facilitator"]:
    if c in df.columns: df[c] = pd.to_numeric(df[c], errors="coerce")
if "segment" in df.columns:
    df["segment"] = df["segment"].astype(str).str.lower().str.strip()

# Flags
df["is_fac"]      = (df.get("role_facilitator", 0).fillna(0) > 0).astype(int)
df["any_team"]    = (df.get("num_teams", 0).fillna(0) > 0).astype(int)
df["any_funded"]  = (df.get("num_funded_teams", 0).fillna(0) > 0).astype(int)

# Annotation feature set (counts)
ann_counts = sorted([c for c in df.columns if c.startswith("ann_") and c.endswith("_count")])

# Small helper: Cohen's d for two groups
def cohens_d(a, b):
    a = pd.to_numeric(a, errors="coerce").dropna()
    b = pd.to_numeric(b, errors="coerce").dropna()
    if len(a) < 3 or len(b) < 3: return np.nan
    na, nb = len(a), len(b)
    va, vb = a.var(ddof=1), b.var(ddof=1)
    pooled = np.sqrt(((na-1)*va + (nb-1)*vb) / (na+nb-2)) if (na+nb-2)>0 and va>=0 and vb>=0 else np.nan
    if pooled == 0 or np.isnan(pooled): return np.nan
    return (a.mean() - b.mean()) / pooled

# ---------------- COMPARISONS ----------------
SEGMENTS = ["first","middle","last"]
CONTROLS = [c for c in ["p_speaking_duration_sec","p_turns"] if c in df.columns]

rows_role = []
rows_fund = []

for seg in SEGMENTS:
    dseg = df[df["segment"]==seg].copy()
    if dseg.empty: continue

    # ---- Facilitator vs Non-facilitator ----
    for f in ann_counts:
        g1 = dseg[dseg["is_fac"]==1][f]
        g0 = dseg[dseg["is_fac"]==0][f]
        mu1, mu0 = g1.mean(skipna=True), g0.mean(skipna=True)
        d   = cohens_d(g1, g0)

        # regression with controls + FE
        rhs = [f]  # (we're testing mean difference, so model f ~ is_fac + controls ... gives p for is_fac)
        # We’ll predict the feature from is_fac to ask “do facilitators exhibit more of this code (by segment)?”
        # (OLS is fine for quick directional tests; counts aren’t huge, and HC3 SE handles heterosked.)
        formula = f"{f} ~ is_fac"
        if CONTROLS:            formula += " + " + " + ".join(CONTROLS)
        if "conference" in dseg.columns: formula += " + C(conference)"
        if "year" in dseg.columns:       formula += " + C(year)"
        m = None
        try:
            m = smf.ols(formula, data=dseg).fit(cov_type="HC3")
            p_isfac = float(m.pvalues.get("is_fac", np.nan))
        except Exception:
            p_isfac = np.nan

        rows_role.append(dict(segment=seg, feature=f, mean_fac=mu1, mean_nonfac=mu0,
                              diff=mu1-mu0, cohens_d=d, p_is_fac=p_isfac,
                              n_fac=int((dseg["is_fac"]==1).sum()), n_nonfac=int((dseg["is_fac"]==0).sum())))

    # ---- Funded vs Not (person-year with any funded team) ----
    for f in ann_counts:
        g1 = dseg[dseg["any_funded"]==1][f]
        g0 = dseg[dseg["any_funded"]==0][f]
        mu1, mu0 = g1.mean(skipna=True), g0.mean(skipna=True)
        d   = cohens_d(g1, g0)

        formula = f"{f} ~ any_funded"
        if CONTROLS:            formula += " + " + " + ".join(CONTROLS)
        if "conference" in dseg.columns: formula += " + C(conference)"
        if "year" in dseg.columns:       formula += " + C(year)"
        m = None
        try:
            m = smf.ols(formula, data=dseg).fit(cov_type="HC3")
            p_funded = float(m.pvalues.get("any_funded", np.nan))
        except Exception:
            p_funded = np.nan

        rows_fund.append(dict(segment=seg, feature=f, mean_funded=mu1, mean_nonfunded=mu0,
                               diff=mu1-mu0, cohens_d=d, p_any_funded=p_funded,
                               n_funded=int((dseg["any_funded"]==1).sum()), n_nonfunded=int((dseg["any_funded"]==0).sum())))

role_df = pd.DataFrame(rows_role)
fund_df = pd.DataFrame(rows_fund)

# Save tidy results
role_csv = OUT_DIR / "temporal_role_differences.csv"
fund_csv = OUT_DIR / "temporal_funded_differences.csv"
role_df.to_csv(role_csv, index=False)
fund_df.to_csv(fund_csv, index=False)
print(f"[OK] wrote {role_csv}  (rows={len(role_df)})")
print(f"[OK] wrote {fund_csv}  (rows={len(fund_df)})")

# ---------------- PLOTS ----------------
def nice_name(s):
    # trim long family names for plotting
    return s.replace("ann_","").replace("_count","")

def plot_top_diffs(df_sub, value_col, p_col, seg, title_prefix, fname_prefix, top=12):
    if df_sub.empty: return
    # rank by absolute difference; keep top
    show = (df_sub
            .assign(absdiff=lambda x: x[value_col].abs())
            .sort_values("absdiff", ascending=False)
            .head(top)
            .sort_values(value_col))
    if show.empty: return
    fig, ax = plt.subplots(figsize=(12, max(4.5, 0.45*len(show))))
    ylab = [nice_name(f) for f in show["feature"]]
    ax.barh(ylab, show[value_col].values, alpha=0.9)
    # mark significance stars at bar ends
    for yi, (val, p) in enumerate(zip(show[value_col].values, show[p_col].values)):
        star = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        ax.text(val + (0.02*show["absdiff"].max() if val>=0 else -0.02*show["absdiff"].max()),
                yi, star, va="center", ha="left" if val>=0 else "right", fontsize=11)
    ax.axvline(0, color="k", lw=1)
    ax.set_xlabel("Difference in mean counts (group A − group B)")
    ax.set_title(f"{title_prefix} — {seg.title()} (bars = group difference, stars = p-value)")
    fig.tight_layout()
    out = OUT_DIR / f"{fname_prefix}_{seg}.png"
    fig.savefig(out, dpi=170)
    plt.close(fig)
    print(f"[OK] wrote {out}")

# Role: facilitators (mean_fac − mean_nonfac)
for seg in SEGMENTS:
    sub = role_df[role_df["segment"]==seg].copy()
    plot_top_diffs(sub, value_col="diff", p_col="p_is_fac",
                   seg=seg,
                   title_prefix="Facilitator vs Non-facilitator (mean annotation counts)",
                   fname_prefix="role_diff_bars")

# Funded: funded − nonfunded
for seg in SEGMENTS:
    sub = fund_df[fund_df["segment"]==seg].copy()
    plot_top_diffs(sub, value_col="diff", p_col="p_any_funded",
                   seg=seg,
                   title_prefix="Funded vs Non-funded (mean annotation counts)",
                   fname_prefix="funded_diff_bars")

# ---------------- QUICK READ-OUT ----------------
def quick_summary(tab, group_cols, diff_col, p_col, k=5):
    lines = []
    for seg in SEGMENTS:
        sub = tab[tab["segment"]==seg].copy()
        if sub.empty: continue
        top = (sub.assign(absdiff=lambda x: x[diff_col].abs())
                   .sort_values([p_col,"absdiff"], ascending=[True,False])
                   .head(k))
        if top.empty: continue
        lines.append(f"\n[{seg.upper()}] strongest differences:")
        for _,r in top.iterrows():
            feat = nice_name(r["feature"])
            star = "***" if r[p_col] < 0.001 else "**" if r[p_col] < 0.01 else "*" if r[p_col] < 0.05 else "ns"
            lines.append(f"  - {feat:35s} diff={r[diff_col]: .3f}  d={r.get('cohens_d',np.nan): .2f}  p={r[p_col]:.3g} ({star})")
    return "\n".join(lines)

print(quick_summary(role_df, ["segment"], "diff", "p_is_fac"))
print(quick_summary(fund_df, ["segment"], "diff", "p_any_funded"))

[OK] wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal_role_differences.csv  (rows=27)
[OK] wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal_funded_differences.csv  (rows=27)
[OK] wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/role_diff_bars_first.png
[OK] wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/role_diff_bars_middle.png
[OK] wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/role_diff_bars_last.png
[OK] wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/funded_diff_bars_first.png
[OK] wrote /Users/maxc

In [2]:
# ============================================================
# ON-TEAM vs NOT-ON-TEAM (new comparison)
# ============================================================

rows_team = []
for seg in SEGMENTS:
    dseg = df[df["segment"]==seg].copy()
    if dseg.empty: continue

    for f in ann_counts:
        g1 = dseg[dseg["any_team"]==1][f]
        g0 = dseg[dseg["any_team"]==0][f]
        mu1, mu0 = g1.mean(skipna=True), g0.mean(skipna=True)
        d = cohens_d(g1, g0)

        formula = f"{f} ~ any_team"
        if CONTROLS: formula += " + " + " + ".join(CONTROLS)
        if "conference" in dseg.columns: formula += " + C(conference)"
        if "year" in dseg.columns: formula += " + C(year)"
        try:
            m = smf.ols(formula, data=dseg).fit(cov_type="HC3")
            p_team = float(m.pvalues.get("any_team", np.nan))
        except Exception:
            p_team = np.nan

        rows_team.append(dict(segment=seg, feature=f,
                              mean_team=mu1, mean_nonteam=mu0,
                              diff=mu1-mu0, cohens_d=d, p_any_team=p_team,
                              n_team=int((dseg["any_team"]==1).sum()),
                              n_nonteam=int((dseg["any_team"]==0).sum())))

team_df = pd.DataFrame(rows_team)
team_csv = OUT_DIR / "temporal_team_differences.csv"
team_df.to_csv(team_csv, index=False)
print(f"[OK] wrote {team_csv} (rows={len(team_df)})")

for seg in SEGMENTS:
    sub = team_df[team_df["segment"]==seg].copy()
    plot_top_diffs(sub, value_col="diff", p_col="p_any_team",
                   seg=seg,
                   title_prefix="On-Team vs Not-On-Team (mean annotation counts)",
                   fname_prefix="team_diff_bars")

print(quick_summary(team_df, ["segment"], "diff", "p_any_team"))

[OK] wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/temporal_team_differences.csv (rows=27)
[OK] wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/team_diff_bars_first.png
[OK] wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/team_diff_bars_middle.png
[OK] wrote /Users/maxchalekson/Projects/NICO-Research/NICO_human-gemini/facilitator-identification/temporal-facilitator-significant/team_diff_bars_last.png

[FIRST] strongest differences:
  - coordination_and_decision_practices diff=-3.517  d=-0.40  p=6.64e-08 (***)
  - participation_dynamics              diff=-2.697  d=-0.38  p=1.45e-06 (***)
  - knowledge_sharing                   diff= 1.846  d= 0.33  p=1.01e-05 (***)
  - relational_climate                  diff=-2.915  d=-0.33  p=2.36e-05 (***)
  - idea_